In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918.sha256.txt
/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918-STATE.json
/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918/DASO-Net-AFTER-SEED2-HANDOFF-20260918/README_HANDOFF.txt
/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918/DASO-Net-AFTER-SEED2-HANDOFF-20260918/RESTORE_AFTER_SEED2.py
/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918/DASO-Net-AFTER-SEED2-HANDOFF-20260918/AFTER_SEED2_STATE.json
/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918/DASO-Net-AFTER-SEED2-HANDOFF-20260918/HANDOFF_MANIFEST.json
/kaggle/input/datasets/abinavprabhuv/dasonet-after-seed2-handoff/DASO-Net-AFTER-SEED2-HANDOFF-20260918/DASO-Net-AFTER-SEE

In [2]:
# ============================================================
# DASO-Net — AFTER-SEED2 RESTORE + FINAL SEED-3 READINESS AUDIT
#
# FRESH NOTEBOOK
#
# PURPOSE
# -------
# Restore the compact After-Seed2 handoff into /kaggle/working
# and verify that:
#
#   - handoff manifest hashes are intact
#   - Seed 1 terminal is intact
#   - Seed 2 terminal is intact
#   - Seed 2 final QC is intact/PASS
#   - validated decoder is intact
#   - frozen final recipe is present
#   - Seed 1/2/3 configs differ only in seed + experiment name
#   - Seed 3 = 252916394
#   - Seed 3 output does not exist
#   - runtime = exactly 2 x Tesla T4
#
# NO TRAINING.
# ============================================================

from __future__ import annotations

import copy
import hashlib
import json
import shutil
from pathlib import Path

import torch


# ============================================================
# 1. EXPECTED SCIENTIFIC IDENTITY
# ============================================================

SEED1 = 1782656963
SEED2 = 901395089
SEED3 = 252916394

SEED3_EXPERIMENT = (
    "dasonet_final_seed03_s252916394"
)

TARGET_SUCCESSFUL_UPDATES = 2750

EXPECTED_PACKAGE_SHA = (
    "3c5c3b9162052cca7ab9cc12c6ee2ffc"
    "6a738b06d77bde7d18fe193be6a6c5dd"
)

EXPECTED_SEED1_TERMINAL_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_TERMINAL_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_SEED2_QC_SUMMARY_SHA = (
    "ddd8798ff38bd54a81dda4c83503e210d76"
    "d3ee4111076884ea7286355fc5bd9"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

HISTORICAL_RECIPE_ZIP_SHA = (
    "b102deb8ec9de4be3816e4450a4dc7b2"
    "f78f5439634cee02fe4656d1a268bf3a"
)


# ============================================================
# 2. INPUT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

HANDOFF_DATASET = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-after-seed2-handoff"
)

PACKAGE_NAME = (
    "DASO-Net-AFTER-SEED2-HANDOFF-20260918"
)

TOP_STATE = (
    HANDOFF_DATASET
    / f"{PACKAGE_NAME}-STATE.json"
)

SHA_TXT = (
    HANDOFF_DATASET
    / f"{PACKAGE_NAME}.sha256.txt"
)

# Kaggle auto-extracted the ZIP, producing:
#
# dataset/
#   PACKAGE_NAME/
#     PACKAGE_NAME/
#       HANDOFF_MANIFEST.json
#       workspace/
#       ...
#
HANDOFF_ROOT = (
    HANDOFF_DATASET
    / PACKAGE_NAME
    / PACKAGE_NAME
)

HANDOFF_MANIFEST = (
    HANDOFF_ROOT
    / "HANDOFF_MANIFEST.json"
)

INNER_STATE = (
    HANDOFF_ROOT
    / "AFTER_SEED2_STATE.json"
)

README = (
    HANDOFF_ROOT
    / "README_HANDOFF.txt"
)

RESTORE_SCRIPT = (
    HANDOFF_ROOT
    / "RESTORE_AFTER_SEED2.py"
)

SOURCE_WORKSPACE = (
    HANDOFF_ROOT
    / "workspace"
)


# ============================================================
# 3. DESTINATION
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED2-RESTORE-AUDIT-v1.json"
)


# ============================================================
# 4. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def copy_tree_fresh(
    src: Path,
    dst: Path,
) -> None:

    if dst.exists():

        raise RuntimeError(
            "Fresh restore destination already exists:\n"
            f"{dst}\n\n"
            "STOP rather than overwriting previous state."
        )

    shutil.copytree(
        src,
        dst,
    )


def copy_file_fresh(
    src: Path,
    dst: Path,
) -> None:

    if dst.exists():

        raise RuntimeError(
            "Fresh restore destination already exists:\n"
            f"{dst}\n\n"
            "STOP rather than overwriting previous state."
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        dst,
    )


def config_without_identity(
    cfg: dict,
) -> dict:

    out = copy.deepcopy(
        cfg
    )

    out.pop(
        "seed",
        None,
    )

    out.pop(
        "experiment_name",
        None,
    )

    return out


# ============================================================
# 5. INITIAL INPUT AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — AFTER-SEED2 RESTORE + SEED-3 READINESS"
)

print(
    "============================================================"
)


required_inputs = [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        HANDOFF_DATASET,
        "After-Seed2 Kaggle dataset",
    ),
    (
        TOP_STATE,
        "Top-level handoff state",
    ),
    (
        SHA_TXT,
        "Original ZIP SHA record",
    ),
    (
        HANDOFF_ROOT,
        "Auto-extracted handoff root",
    ),
    (
        HANDOFF_MANIFEST,
        "Handoff file manifest",
    ),
    (
        INNER_STATE,
        "Internal After-Seed2 state",
    ),
    (
        README,
        "Handoff README",
    ),
    (
        RESTORE_SCRIPT,
        "Handoff restore script",
    ),
    (
        SOURCE_WORKSPACE,
        "Handoff workspace",
    ),
]


for path, label in required_inputs:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 6. PROJECT DATA PRESENCE
# ============================================================

project_required = [
    PROJECT_ROOT
    / "configs"
    / "renderer_v3_1.json",

    PROJECT_ROOT
    / "src"
    / "dasonet"
    / "synth"
    / "renderer.py",

    PROJECT_ROOT
    / "data"
    / "mp_morph"
    / "v1"
    / "morphology_manifest_v1.csv",

    PROJECT_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv",
]


for path in project_required:

    require(
        path,
        "Frozen project component",
    )


print()
print(
    "Frozen project critical components: PASS"
)


# ============================================================
# 7. OUTER STATE / PACKAGE PROVENANCE
# ============================================================

outer_state = load_json(
    TOP_STATE
)

inner_state = load_json(
    INNER_STATE
)


if (
    outer_state[
        "package_sha256"
    ]
    != EXPECTED_PACKAGE_SHA
):

    raise RuntimeError(
        "Outer state package SHA provenance mismatch."
    )


sha_text = SHA_TXT.read_text(
    encoding="utf-8"
).strip()


if EXPECTED_PACKAGE_SHA not in sha_text:

    raise RuntimeError(
        "SHA256 text file does not contain expected package SHA."
    )


if (
    outer_state[
        "seed1_terminal_sha256"
    ]
    != EXPECTED_SEED1_TERMINAL_SHA
):

    raise RuntimeError(
        "Outer state Seed-1 SHA mismatch."
    )


if (
    outer_state[
        "seed2_terminal_sha256"
    ]
    != EXPECTED_SEED2_TERMINAL_SHA
):

    raise RuntimeError(
        "Outer state Seed-2 SHA mismatch."
    )


if (
    outer_state[
        "seed2_qc_summary_sha256"
    ]
    != EXPECTED_SEED2_QC_SUMMARY_SHA
):

    raise RuntimeError(
        "Outer state Seed-2 QC summary SHA mismatch."
    )


if (
    outer_state[
        "decoder_sha256"
    ]
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Outer state decoder SHA mismatch."
    )


if int(
    outer_state[
        "next_seed"
    ][
        "index"
    ]
) != 3:

    raise RuntimeError(
        "Outer state does not designate Seed 3."
    )


if int(
    outer_state[
        "next_seed"
    ][
        "rng_seed"
    ]
) != SEED3:

    raise RuntimeError(
        "Outer state Seed-3 RNG mismatch."
    )


if (
    outer_state[
        "next_seed"
    ][
        "resume"
    ]
    != "none"
):

    raise RuntimeError(
        "Outer state Seed-3 resume policy is not NONE."
    )


if int(
    outer_state[
        "next_seed"
    ][
        "target_successful_optimizer_updates"
    ]
) != TARGET_SUCCESSFUL_UPDATES:

    raise RuntimeError(
        "Outer state training budget is not 2750."
    )


if (
    inner_state[
        "historical_original_recipe_zip_sha256"
    ]
    != HISTORICAL_RECIPE_ZIP_SHA
):

    raise RuntimeError(
        "Historical recipe-ZIP provenance mismatch."
    )


print()
print(
    "Original handoff ZIP SHA provenance: PASS"
)

print(
    "Seed-1 terminal provenance: PASS"
)

print(
    "Seed-2 terminal provenance: PASS"
)

print(
    "Seed-2 QC provenance: PASS"
)

print(
    "Seed-3 frozen identity: PASS"
)


# ============================================================
# 8. VERIFY EVERY FILE IN HANDOFF MANIFEST
# ============================================================

print()
print(
    "============================================================"
)

print(
    "HANDOFF FILE-BY-FILE SHA AUDIT"
)

print(
    "============================================================"
)


manifest = load_json(
    HANDOFF_MANIFEST
)

records = manifest[
    "files"
]


if int(
    manifest[
        "file_count"
    ]
) != len(
    records
):

    raise RuntimeError(
        "Handoff manifest file count mismatch."
    )


manifest_verified = 0


for record in records:

    relative_path = record[
        "relative_path"
    ]

    path = (
        HANDOFF_ROOT
        / relative_path
    )

    require(
        path,
        f"Manifest file {relative_path}",
    )


    actual_size = (
        path.stat().st_size
    )

    expected_size = int(
        record[
            "size_bytes"
        ]
    )


    if (
        actual_size
        != expected_size
    ):

        raise RuntimeError(
            "Manifest size mismatch:\n"
            f"{relative_path}\n"
            f"Expected: {expected_size}\n"
            f"Actual:   {actual_size}"
        )


    actual_sha = sha256_file(
        path
    )

    expected_sha = record[
        "sha256"
    ]


    if actual_sha != expected_sha:

        raise RuntimeError(
            "Manifest SHA mismatch:\n"
            f"{relative_path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )


    manifest_verified += 1


print(
    f"Manifest files verified: {manifest_verified}"
)

print(
    "File-by-file SHA audit: PASS"
)


# ============================================================
# 9. CRITICAL SOURCE PATHS INSIDE HANDOFF
# ============================================================

SRC_CODE = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5B-KaggleTraining"
)

SRC_MULTISEED = (
    SOURCE_WORKSPACE
    / "DASO-Net-FinalMultiSeed-v1"
)

SRC_FREEZE = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

SRC_SEED1_RUN = (
    SOURCE_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

SRC_SEED2_RUN = (
    SOURCE_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)

SRC_SEED1_QC = (
    SOURCE_WORKSPACE
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)

SRC_SEED2_QC = (
    SOURCE_WORKSPACE
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
)

SRC_SEED1_TERMINAL = (
    SRC_SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SRC_SEED2_TERMINAL = (
    SRC_SEED2_RUN
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SRC_SEED1_TRAINING_MANIFEST = (
    SRC_SEED1_RUN
    / "FINAL_SEED01_TRAINING_MANIFEST.json"
)

SRC_SEED2_TRAINING_MANIFEST = (
    SRC_SEED2_RUN
    / "FINAL_SEED02_TRAINING_MANIFEST.json"
)

SRC_SEED2_EXACT_MANIFEST = (
    SRC_SEED2_RUN
    / "FINAL_SEED02_EXACT_UPDATE_COMPLETION.json"
)

SRC_SEED1_QC_SUMMARY = (
    SRC_SEED1_QC
    / "FINAL_SEED01_QC_SUMMARY.json"
)

SRC_SEED2_QC_SUMMARY = (
    SRC_SEED2_QC
    / "FINAL_SEED02_QC_SUMMARY.json"
)

SRC_PLAN = (
    SRC_MULTISEED
    / "FINAL_MULTISEED_PLAN_v1.json"
)

SRC_SEED1_CONFIG = (
    SRC_MULTISEED
    / "configs"
    / "final_seed01_s1782656963.json"
)

SRC_SEED2_CONFIG = (
    SRC_MULTISEED
    / "configs"
    / "final_seed02_s901395089.json"
)

SRC_SEED3_CONFIG = (
    SRC_MULTISEED
    / "configs"
    / "final_seed03_s252916394.json"
)

SRC_DECODER = (
    SRC_CODE
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

SRC_TRAINER = (
    SRC_CODE
    / "scripts"
    / "train_phase5b.py"
)

SRC_LAUNCHER = (
    SRC_CODE
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

SRC_EVALUATOR = (
    SRC_CODE
    / "scripts"
    / "evaluate_phase5c.py"
)

SRC_RECIPE_JSON = (
    SRC_FREEZE
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json"
)

SRC_RECIPE_MANIFEST = (
    SRC_FREEZE
    / "FREEZE_FILE_MANIFEST.json"
)


critical_sources = [
    (SRC_CODE, "Validated code tree"),
    (SRC_MULTISEED, "Frozen multi-seed directory"),
    (SRC_FREEZE, "Frozen training-recipe directory"),

    (SRC_SEED1_TERMINAL, "Seed-1 terminal"),
    (SRC_SEED2_TERMINAL, "Seed-2 terminal"),

    (
        SRC_SEED1_TRAINING_MANIFEST,
        "Seed-1 training manifest",
    ),
    (
        SRC_SEED2_TRAINING_MANIFEST,
        "Seed-2 training manifest",
    ),
    (
        SRC_SEED2_EXACT_MANIFEST,
        "Seed-2 exact-update manifest",
    ),

    (
        SRC_SEED1_QC_SUMMARY,
        "Seed-1 QC summary",
    ),
    (
        SRC_SEED2_QC_SUMMARY,
        "Seed-2 QC summary",
    ),

    (SRC_PLAN, "Frozen multi-seed plan"),

    (SRC_SEED1_CONFIG, "Seed-1 config"),
    (SRC_SEED2_CONFIG, "Seed-2 config"),
    (SRC_SEED3_CONFIG, "Seed-3 config"),

    (SRC_DECODER, "Validated decoder"),
    (SRC_TRAINER, "Validated trainer"),
    (SRC_LAUNCHER, "Validated launcher"),
    (SRC_EVALUATOR, "Validated evaluator"),

    (SRC_RECIPE_JSON, "Frozen recipe JSON"),
    (
        SRC_RECIPE_MANIFEST,
        "Frozen recipe file manifest",
    ),
]


for path, label in critical_sources:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 10. VERIFY CRITICAL HASHES BEFORE RESTORE
# ============================================================

if (
    sha256_file(
        SRC_SEED1_TERMINAL
    )
    != EXPECTED_SEED1_TERMINAL_SHA
):

    raise RuntimeError(
        "Seed-1 terminal SHA failure."
    )


if (
    sha256_file(
        SRC_SEED2_TERMINAL
    )
    != EXPECTED_SEED2_TERMINAL_SHA
):

    raise RuntimeError(
        "Seed-2 terminal SHA failure."
    )


if (
    sha256_file(
        SRC_SEED2_QC_SUMMARY
    )
    != EXPECTED_SEED2_QC_SUMMARY_SHA
):

    raise RuntimeError(
        "Seed-2 QC summary SHA failure."
    )


if (
    sha256_file(
        SRC_DECODER
    )
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Decoder SHA failure."
    )


if (
    sha256_file(
        SRC_PLAN
    )
    != outer_state[
        "multiseed_plan_sha256"
    ]
):

    raise RuntimeError(
        "Multi-seed plan SHA failure."
    )


if (
    sha256_file(
        SRC_SEED3_CONFIG
    )
    != outer_state[
        "seed3_config_sha256"
    ]
):

    raise RuntimeError(
        "Seed-3 config SHA failure."
    )


print()
print(
    "Seed-1 terminal SHA: PASS"
)

print(
    "Seed-2 terminal SHA: PASS"
)

print(
    "Seed-2 QC summary SHA: PASS"
)

print(
    "Decoder SHA: PASS"
)

print(
    "Multi-seed plan SHA: PASS"
)

print(
    "Seed-3 config SHA: PASS"
)


# ============================================================
# 11. VERIFY COMPLETED SEED TRAINING / QC STATUS
# ============================================================

seed1_training = load_json(
    SRC_SEED1_TRAINING_MANIFEST
)

seed2_training = load_json(
    SRC_SEED2_TRAINING_MANIFEST
)

seed2_exact = load_json(
    SRC_SEED2_EXACT_MANIFEST
)

seed1_qc = load_json(
    SRC_SEED1_QC_SUMMARY
)

seed2_qc = load_json(
    SRC_SEED2_QC_SUMMARY
)


if int(
    seed1_training[
        "successful_optimizer_updates"
    ]
) != TARGET_SUCCESSFUL_UPDATES:

    raise RuntimeError(
        "Seed-1 training record is not at 2750."
    )


if int(
    seed2_training[
        "successful_optimizer_updates"
    ]
) != TARGET_SUCCESSFUL_UPDATES:

    raise RuntimeError(
        "Seed-2 training record is not at 2750."
    )


if int(
    seed2_exact[
        "final_successful_optimizer_updates"
    ]
) != TARGET_SUCCESSFUL_UPDATES:

    raise RuntimeError(
        "Seed-2 exact-update record is not at 2750."
    )


if (
    seed2_qc.get(
        "status"
    )
    != "PASS"
):

    raise RuntimeError(
        "Seed-2 final QC status is not PASS."
    )


if not all(
    seed2_qc[
        "sanity_flags"
    ].values()
):

    raise RuntimeError(
        "Seed-2 final QC sanity flags are not all PASS."
    )


if (
    "sanity_flags"
    in seed1_qc
):

    if not all(
        seed1_qc[
            "sanity_flags"
        ].values()
    ):

        raise RuntimeError(
            "Seed-1 QC contains failed sanity flags."
        )


print()
print(
    "Seed-1 exact training budget: PASS"
)

print(
    "Seed-2 exact training budget: PASS"
)

print(
    "Seed-1 QC integrity: PASS"
)

print(
    "Seed-2 final QC: PASS"
)


# ============================================================
# 12. VERIFY FINAL CONFIGS DIFFER ONLY IN SEED + RUN NAME
# ============================================================

cfg1 = load_json(
    SRC_SEED1_CONFIG
)

cfg2 = load_json(
    SRC_SEED2_CONFIG
)

cfg3 = load_json(
    SRC_SEED3_CONFIG
)


if int(
    cfg1[
        "seed"
    ]
) != SEED1:

    raise RuntimeError(
        "Seed-1 config RNG mismatch."
    )


if int(
    cfg2[
        "seed"
    ]
) != SEED2:

    raise RuntimeError(
        "Seed-2 config RNG mismatch."
    )


if int(
    cfg3[
        "seed"
    ]
) != SEED3:

    raise RuntimeError(
        "Seed-3 config RNG mismatch."
    )


if (
    cfg3[
        "experiment_name"
    ]
    != SEED3_EXPERIMENT
):

    raise RuntimeError(
        "Seed-3 experiment-name mismatch."
    )


base1 = config_without_identity(
    cfg1
)

base2 = config_without_identity(
    cfg2
)

base3 = config_without_identity(
    cfg3
)


if (
    base1 != base2
    or base1 != base3
):

    raise RuntimeError(
        "Final seed configs differ in scientific settings.\n"
        "Expected only seed + experiment_name differences."
    )


pcc_center = (
    cfg3
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "Seed-3 PCC-center unexpectedly enabled."
    )


print()
print(
    "Seed configs differ only by identity fields: PASS"
)

print(
    "Seed-3 RNG 252916394: PASS"
)

print(
    "Seed-3 experiment name: PASS"
)

print(
    "Seed-3 PCC-center disabled: PASS"
)


# ============================================================
# 13. VERIFY FROZEN PLAN
# ============================================================

plan = load_json(
    SRC_PLAN
)


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != TARGET_SUCCESSFUL_UPDATES:

    raise RuntimeError(
        "Frozen plan does not specify 2750 updates."
    )


seed3_plan_entry = None


for entry in plan[
    "final_seed_configs"
]:

    if int(
        entry[
            "seed_index"
        ]
    ) == 3:

        seed3_plan_entry = entry
        break


if seed3_plan_entry is None:

    raise RuntimeError(
        "Seed-3 entry missing from frozen plan."
    )


if int(
    seed3_plan_entry[
        "seed"
    ]
) != SEED3:

    raise RuntimeError(
        "Frozen plan Seed-3 RNG mismatch."
    )


if (
    seed3_plan_entry[
        "experiment_name"
    ]
    != SEED3_EXPERIMENT
):

    raise RuntimeError(
        "Frozen plan Seed-3 experiment mismatch."
    )


print(
    "Frozen multi-seed plan Seed-3 entry: PASS"
)


# ============================================================
# 14. GPU RUNTIME AUDIT
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable. Select GPU accelerator."
    )


gpu_count = (
    torch.cuda.device_count()
)


if gpu_count != 2:

    raise RuntimeError(
        f"Expected exactly 2 GPUs; found {gpu_count}."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(
        gpu_count
    )
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        "Final Seed-3 protocol requires 2x Tesla T4.\n"
        f"Detected: {gpu_names}"
    )


print()
print(
    "GPU runtime: 2x Tesla T4 — PASS"
)


# ============================================================
# 15. VERIFY SEED-3 OUTPUT ABSENT BEFORE RESTORE
# ============================================================

SEED3_OUTPUT = (
    WORKING
    / "dasonet_phase5b"
    / SEED3_EXPERIMENT
)


if SEED3_OUTPUT.exists():

    raise RuntimeError(
        "Seed-3 output already exists BEFORE restore:\n"
        f"{SEED3_OUTPUT}\n\n"
        "Fresh initialization cannot be guaranteed."
    )


print(
    "Seed-3 output absent before restore: PASS"
)


# ============================================================
# 16. RESTORE WORKSPACE
#
# Fresh notebook policy:
# every handoff workspace top-level entry must be absent before
# copying. We do not silently overwrite anything.
# ============================================================

print()
print(
    "============================================================"
)

print(
    "RESTORING AFTER-SEED2 WORKSPACE"
)

print(
    "============================================================"
)


for src in sorted(
    SOURCE_WORKSPACE.iterdir(),
    key=lambda p: p.name,
):

    dst = (
        WORKING
        / src.name
    )


    if src.is_dir():

        copy_tree_fresh(
            src,
            dst,
        )

    else:

        copy_file_fresh(
            src,
            dst,
        )


    print(
        f"PASS  restored {src.name}"
    )


# ============================================================
# 17. POST-RESTORE PATHS
# ============================================================

DST_CODE = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

DST_MULTISEED = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

DST_FREEZE = (
    WORKING
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

DST_SEED1_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

DST_SEED2_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

DST_SEED2_QC_SUMMARY = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
    / "FINAL_SEED02_QC_SUMMARY.json"
)

DST_SEED3_CONFIG = (
    DST_MULTISEED
    / "configs"
    / "final_seed03_s252916394.json"
)

DST_PLAN = (
    DST_MULTISEED
    / "FINAL_MULTISEED_PLAN_v1.json"
)

DST_DECODER = (
    DST_CODE
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

DST_TRAINER = (
    DST_CODE
    / "scripts"
    / "train_phase5b.py"
)

DST_LAUNCHER = (
    DST_CODE
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

DST_EVALUATOR = (
    DST_CODE
    / "scripts"
    / "evaluate_phase5c.py"
)

DST_RECIPE_JSON = (
    DST_FREEZE
    / "FINAL_TRAINING_RECIPE_FREEZE_v1.json"
)

DST_RECIPE_MANIFEST = (
    DST_FREEZE
    / "FREEZE_FILE_MANIFEST.json"
)


post_required = [
    (
        DST_CODE,
        "Restored validated code",
    ),
    (
        DST_MULTISEED,
        "Restored multi-seed plan",
    ),
    (
        DST_FREEZE,
        "Restored frozen recipe",
    ),
    (
        DST_SEED1_TERMINAL,
        "Restored Seed-1 terminal",
    ),
    (
        DST_SEED2_TERMINAL,
        "Restored Seed-2 terminal",
    ),
    (
        DST_SEED2_QC_SUMMARY,
        "Restored Seed-2 QC summary",
    ),
    (
        DST_SEED3_CONFIG,
        "Restored Seed-3 config",
    ),
    (
        DST_DECODER,
        "Restored decoder",
    ),
    (
        DST_TRAINER,
        "Restored trainer",
    ),
    (
        DST_LAUNCHER,
        "Restored launcher",
    ),
    (
        DST_EVALUATOR,
        "Restored evaluator",
    ),
    (
        DST_RECIPE_JSON,
        "Restored frozen recipe JSON",
    ),
    (
        DST_RECIPE_MANIFEST,
        "Restored recipe manifest",
    ),
]


for path, label in post_required:

    require(
        path,
        label,
    )


# ============================================================
# 18. POST-RESTORE HASH AUDIT
# ============================================================

if (
    sha256_file(
        DST_SEED1_TERMINAL
    )
    != EXPECTED_SEED1_TERMINAL_SHA
):

    raise RuntimeError(
        "POST-RESTORE Seed-1 SHA failure."
    )


if (
    sha256_file(
        DST_SEED2_TERMINAL
    )
    != EXPECTED_SEED2_TERMINAL_SHA
):

    raise RuntimeError(
        "POST-RESTORE Seed-2 SHA failure."
    )


if (
    sha256_file(
        DST_SEED2_QC_SUMMARY
    )
    != EXPECTED_SEED2_QC_SUMMARY_SHA
):

    raise RuntimeError(
        "POST-RESTORE Seed-2 QC SHA failure."
    )


if (
    sha256_file(
        DST_DECODER
    )
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "POST-RESTORE decoder SHA failure."
    )


if (
    sha256_file(
        DST_PLAN
    )
    != outer_state[
        "multiseed_plan_sha256"
    ]
):

    raise RuntimeError(
        "POST-RESTORE plan SHA failure."
    )


if (
    sha256_file(
        DST_SEED3_CONFIG
    )
    != outer_state[
        "seed3_config_sha256"
    ]
):

    raise RuntimeError(
        "POST-RESTORE Seed-3 config SHA failure."
    )


print()
print(
    "Restored Seed-1 terminal SHA: PASS"
)

print(
    "Restored Seed-2 terminal SHA: PASS"
)

print(
    "Restored Seed-2 QC SHA: PASS"
)

print(
    "Restored decoder SHA: PASS"
)

print(
    "Restored multi-seed plan SHA: PASS"
)

print(
    "Restored Seed-3 config SHA: PASS"
)


# ============================================================
# 19. FINAL SEED-3 FRESHNESS CHECK
# ============================================================

if SEED3_OUTPUT.exists():

    raise RuntimeError(
        "Seed-3 output appeared during restore unexpectedly."
    )


restored_cfg3 = load_json(
    DST_SEED3_CONFIG
)


if int(
    restored_cfg3[
        "seed"
    ]
) != SEED3:

    raise RuntimeError(
        "Restored Seed-3 RNG mismatch."
    )


if (
    restored_cfg3[
        "experiment_name"
    ]
    != SEED3_EXPERIMENT
):

    raise RuntimeError(
        "Restored Seed-3 experiment mismatch."
    )


print()
print(
    "Seed-3 output still absent: PASS"
)

print(
    "Seed-3 fresh initialization condition: PASS"
)

print(
    "Seed-3 resume policy: NONE"
)


# ============================================================
# 20. WRITE RESTORE AUDIT
# ============================================================

audit = {
    "audit_name":
        "DASO-Net AFTER-SEED2 restore + Seed-3 readiness v1",

    "handoff_source":
        str(
            HANDOFF_ROOT
        ),

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "package_provenance": {
        "original_zip_sha256":
            EXPECTED_PACKAGE_SHA,

        "raw_zip_visible_in_kaggle_mount":
            (
                HANDOFF_DATASET
                / f"{PACKAGE_NAME}.zip"
            ).exists(),

        "sha_record_verified":
            True,

        "handoff_manifest_file_count":
            len(
                records
            ),

        "handoff_manifest_verified":
            True,
    },

    "seed1": {
        "rng_seed":
            SEED1,

        "successful_optimizer_updates":
            2750,

        "terminal_sha256":
            EXPECTED_SEED1_TERMINAL_SHA,
    },

    "seed2": {
        "rng_seed":
            SEED2,

        "successful_optimizer_updates":
            2750,

        "terminal_sha256":
            EXPECTED_SEED2_TERMINAL_SHA,

        "qc":
            "PASS",

        "qc_summary_sha256":
            EXPECTED_SEED2_QC_SUMMARY_SHA,
    },

    "seed3": {
        "seed_index":
            3,

        "rng_seed":
            SEED3,

        "experiment_name":
            SEED3_EXPERIMENT,

        "initialization":
            "fresh",

        "resume":
            "none",

        "target_successful_optimizer_updates":
            TARGET_SUCCESSFUL_UPDATES,

        "output_absent":
            True,
    },

    "runtime": {
        "gpu_count":
            gpu_count,

        "gpu_names":
            gpu_names,
    },

    "critical_hashes": {
        "decoder":
            sha256_file(
                DST_DECODER
            ),

        "multiseed_plan":
            sha256_file(
                DST_PLAN
            ),

        "seed3_config":
            sha256_file(
                DST_SEED3_CONFIG
            ),

        "seed1_terminal":
            sha256_file(
                DST_SEED1_TERMINAL
            ),

        "seed2_terminal":
            sha256_file(
                DST_SEED2_TERMINAL
            ),

        "seed2_qc_summary":
            sha256_file(
                DST_SEED2_QC_SUMMARY
            ),
    },

    "historical_recipe_zip_sha256":
        HISTORICAL_RECIPE_ZIP_SHA,

    "scientific_settings_identical_across_final_configs":
        True,
}


save_json(
    RESTORE_AUDIT,
    audit,
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "AFTER-SEED2 RESTORE COMPLETE"
)

print(
    "============================================================"
)

print()

print(
    "Seed 1:"
)

print(
    "  status:                  COMPLETE"
)

print(
    "  successful updates:      2750"
)

print(
    "  terminal SHA:            PASS"
)

print()

print(
    "Seed 2:"
)

print(
    "  status:                  COMPLETE"
)

print(
    "  successful updates:      2750"
)

print(
    "  terminal SHA:            PASS"
)

print(
    "  fixed QC:                PASS"
)

print()

print(
    "Seed 3:"
)

print(
    "  index:                   3 / 3"
)

print(
    "  RNG seed:                252916394"
)

print(
    "  initialization:          FRESH"
)

print(
    "  resume:                  NONE"
)

print(
    "  target successful step:  2750"
)

print(
    "  output absent:           YES"
)

print()

print(
    "Runtime:"
)

print(
    "  GPUs:                    ",
    gpu_names,
)

print()

print(
    "Restore audit:"
)

print(
    RESTORE_AUDIT
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED2 RESTORE + SEED-3 READINESS: PASS"
)

print(
    "============================================================"
)

print(
    "NO TRAINING WAS STARTED."
)

print(
    "STOP HERE."
)

print(
    "Paste this final output before Final Seed 3 launch."
)

DASO-Net — AFTER-SEED2 RESTORE + SEED-3 READINESS
PASS  Frozen project dataset
PASS  After-Seed2 Kaggle dataset
PASS  Top-level handoff state
PASS  Original ZIP SHA record
PASS  Auto-extracted handoff root
PASS  Handoff file manifest
PASS  Internal After-Seed2 state
PASS  Handoff README
PASS  Handoff restore script
PASS  Handoff workspace

Frozen project critical components: PASS

Original handoff ZIP SHA provenance: PASS
Seed-1 terminal provenance: PASS
Seed-2 terminal provenance: PASS
Seed-2 QC provenance: PASS
Seed-3 frozen identity: PASS

HANDOFF FILE-BY-FILE SHA AUDIT
Manifest files verified: 84
File-by-file SHA audit: PASS
PASS  Validated code tree
PASS  Frozen multi-seed directory
PASS  Frozen training-recipe directory
PASS  Seed-1 terminal
PASS  Seed-2 terminal
PASS  Seed-1 training manifest
PASS  Seed-2 training manifest
PASS  Seed-2 exact-update manifest
PASS  Seed-1 QC summary
PASS  Seed-2 QC summary
PASS  Frozen multi-seed plan
PASS  Seed-1 config
PASS  Seed-2 config
PASS  

In [3]:
# ============================================================
# DASO-Net — FINAL MULTI-SEED TRAINING
# FINAL SEED 3 / 3
#
# RNG seed:
#   252916394
#
# FROZEN SCIENTIFIC RECIPE
# ------------------------
# fresh initialization
# target = EXACTLY 2750 successful optimizer updates
# native tile = 1024
# 2 x Tesla T4
# synthetic batch/GPU = 1
# anchor pair batch/GPU = 2
# gradient accumulation = 4
# effective synthetic batch = 8
# synthetic workers/GPU = 2
# anchor workers/GPU = 1
# LR = 3e-4
# warmup = 50 successful optimizer updates
# constant post-warmup LR
# AMP FP16 initial scale = 32
# original PCC retained
# CRPA retained
# PCC-center disabled
#
# IMPORTANT
# ---------
# - Initial run is FRESH: --resume none
# - No early stopping
# - best.pt is NOT used for final reporting
# - terminal model = checkpoint at exactly 2750 successful
#   optimizer updates
# - if AMP skips occur, bounded compensation is performed
#   automatically
# - external data are NOT used
#
# After PASS:
#   STOP.
#   Do NOT launch any new training.
#   Next = fixed Seed-3 QC.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import time
from pathlib import Path

import torch


# ============================================================
# 1. FROZEN CONSTANTS
# ============================================================

SEED_INDEX = 3

RNG_SEED = 252916394

EXPERIMENT = (
    "dasonet_final_seed03_s252916394"
)

TARGET_OPTIMIZER_STEP = 2750

INITIAL_EPOCHS = 11

INITIAL_MICROSTEPS_PER_EPOCH = 1000

ACCUM = 4

EXPECTED_UPDATES_PER_FULL_BLOCK = (
    INITIAL_MICROSTEPS_PER_EPOCH // ACCUM
)

EXPECTED_INITIAL_UPDATE_ATTEMPTS = (
    INITIAL_EPOCHS
    * EXPECTED_UPDATES_PER_FULL_BLOCK
)

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe573"
    "8d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# 2. PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

WORKING = Path(
    "/kaggle/working"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

PLAN_ROOT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

PLAN_JSON = (
    PLAN_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

CONFIG = (
    PLAN_ROOT
    / "configs"
    / "final_seed03_s252916394.json"
)

RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED2-RESTORE-AUDIT-v1.json"
)

SEED1_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED2_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

RUN_ROOT = (
    WORKING
    / "dasonet_phase5b"
    / EXPERIMENT
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

RUN_STATUS = (
    RUN_ROOT
    / "run_status.json"
)

RESOLVED_CONFIG = (
    RUN_ROOT
    / "resolved_config.json"
)

HISTORY = (
    RUN_ROOT
    / "history.csv"
)

INITIAL_STATUS_COPY = (
    RUN_ROOT
    / "run_status_initial_11blocks.json"
)

INITIAL_RESOLVED_COPY = (
    RUN_ROOT
    / "resolved_config_initial_11blocks.json"
)

INITIAL_HISTORY_COPY = (
    RUN_ROOT
    / "history_initial_11blocks.csv"
)

EXACT_UPDATE_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED03_EXACT_UPDATE_COMPLETION.json"
)

FINAL_TRAINING_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED03_TRAINING_MANIFEST.json"
)


# ============================================================
# 3. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def trusted_load(
    path: Path,
):

    return torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )


def snapshot_records():
    """
    Read every Seed-3 snapshot and return chronological metadata.
    """

    records = []

    if not SNAPSHOT_DIR.exists():

        return records


    for path in SNAPSHOT_DIR.glob(
        "epoch_*_opt_*.pt"
    ):

        payload = trusted_load(
            path
        )

        records.append(
            {
                "path":
                    path,

                "epoch":
                    int(
                        payload.get(
                            "epoch",
                            -1,
                        )
                    ),

                "optimizer_step":
                    int(
                        payload.get(
                            "optimizer_step",
                            -1,
                        )
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


    records.sort(
        key=lambda x: (
            x[
                "epoch"
            ],
            x[
                "optimizer_step"
            ],
        )
    )

    return records


def verify_constant_lr(
    checkpoint: Path,
) -> list[float]:

    payload = trusted_load(
        checkpoint
    )

    optimizer_state = payload.get(
        "optimizer_state_dict",
        {}
    )

    groups = optimizer_state.get(
        "param_groups",
        []
    )


    lrs = [
        float(
            group[
                "lr"
            ]
        )
        for group in groups
        if "lr" in group
    ]


    if not lrs:

        raise RuntimeError(
            "Could not read optimizer learning rate "
            "from checkpoint."
        )


    for lr in lrs:

        if abs(
            lr
            - 3e-4
        ) > 1e-10:

            raise RuntimeError(
                "Unexpected final/completion learning rate.\n"
                f"Observed: {lr}"
            )


    return lrs


# ============================================================
# 4. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL SEED 3 / 3 PREFLIGHT"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        CODE_ROOT,
        "Restored validated code tree",
    ),
    (
        TRAINER,
        "Validated trainer",
    ),
    (
        LAUNCHER,
        "Validated checkpoint launcher",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        PLAN_JSON,
        "Frozen multi-seed plan",
    ),
    (
        CONFIG,
        "Frozen Seed-3 config",
    ),
    (
        RESTORE_AUDIT,
        "After-Seed2 restore audit",
    ),
    (
        SEED1_TERMINAL,
        "Final Seed-1 terminal checkpoint",
    ),
    (
        SEED2_TERMINAL,
        "Final Seed-2 terminal checkpoint",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. VERIFY RESTORED STATE
# ============================================================

restore = load_json(
    RESTORE_AUDIT
)


if int(
    restore[
        "seed3"
    ][
        "rng_seed"
    ]
) != RNG_SEED:

    raise RuntimeError(
        "Restore audit does not designate Seed 3 correctly."
    )


if (
    restore[
        "seed3"
    ][
        "resume"
    ]
    != "none"
):

    raise RuntimeError(
        "Restore audit Seed-3 resume policy is not NONE."
    )


if int(
    restore[
        "seed3"
    ][
        "target_successful_optimizer_updates"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Restore audit target is not 2750."
    )


if (
    sha256_file(
        SEED1_TERMINAL
    )
    != EXPECTED_SEED1_SHA
):

    raise RuntimeError(
        "Seed-1 terminal checkpoint changed after restore."
    )


if (
    sha256_file(
        SEED2_TERMINAL
    )
    != EXPECTED_SEED2_SHA
):

    raise RuntimeError(
        "Seed-2 terminal checkpoint changed after restore."
    )


if (
    sha256_file(
        DECODER
    )
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


print()
print(
    "After-Seed2 restore state: PASS"
)

print(
    "Seed-1 terminal remains intact: PASS"
)

print(
    "Seed-2 terminal remains intact: PASS"
)

print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 6. VERIFY FROZEN MULTI-SEED PLAN + SEED-3 CONFIG
# ============================================================

plan = load_json(
    PLAN_JSON
)

cfg = load_json(
    CONFIG
)


if (
    plan.get(
        "status"
    )
    != "FROZEN_BEFORE_FINAL_TRAINING"
):

    raise RuntimeError(
        "Multi-seed plan status is not frozen."
    )


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Frozen training budget is not 2750."
    )


planned_seed3 = None


for item in plan[
    "final_seed_configs"
]:

    if int(
        item[
            "seed_index"
        ]
    ) == SEED_INDEX:

        planned_seed3 = item
        break


if planned_seed3 is None:

    raise RuntimeError(
        "Seed-3 entry missing from frozen multi-seed plan."
    )


if int(
    planned_seed3[
        "seed"
    ]
) != RNG_SEED:

    raise RuntimeError(
        "Seed-3 RNG mismatch in frozen plan."
    )


if (
    planned_seed3[
        "experiment_name"
    ]
    != EXPERIMENT
):

    raise RuntimeError(
        "Seed-3 experiment mismatch in frozen plan."
    )


actual_config_sha = sha256_file(
    CONFIG
)


if (
    actual_config_sha
    != planned_seed3[
        "config_sha256"
    ]
):

    raise RuntimeError(
        "Seed-3 config SHA mismatch.\n"
        f"Expected: {planned_seed3['config_sha256']}\n"
        f"Actual:   {actual_config_sha}"
    )


if int(
    cfg[
        "seed"
    ]
) != RNG_SEED:

    raise RuntimeError(
        "Seed-3 config RNG mismatch."
    )


if (
    cfg[
        "experiment_name"
    ]
    != EXPERIMENT
):

    raise RuntimeError(
        "Seed-3 config experiment mismatch."
    )


pcc_center = (
    cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center unexpectedly enabled."
    )


print(
    "Frozen multi-seed plan: PASS"
)

print(
    "Seed-3 config SHA: PASS"
)

print(
    "Seed-3 RNG seed: 252916394 — PASS"
)

print(
    "PCC-center disabled: PASS"
)


# ============================================================
# 7. FRESH INITIALIZATION SAFETY
# ============================================================

if RUN_ROOT.exists():

    raise RuntimeError(
        "FINAL SEED-3 OUTPUT DIRECTORY ALREADY EXISTS:\n"
        f"{RUN_ROOT}\n\n"
        "Refusing to start. Fresh initialization cannot "
        "be guaranteed.\n"
        "Do NOT delete anything manually; inspect first."
    )


print()
print(
    "Fresh initialization condition: PASS"
)

print(
    "Seed-3 output directory is absent."
)


# ============================================================
# 8. GPU AUDIT
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


if torch.cuda.device_count() != 2:

    raise RuntimeError(
        f"Expected exactly 2 GPUs; "
        f"found {torch.cuda.device_count()}."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(2)
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        "Frozen runtime requires 2x Tesla T4.\n"
        f"Detected: {gpu_names}"
    )


print(
    "GPU runtime: 2x Tesla T4 — PASS"
)


# ============================================================
# 9. TRAINING CONTRACT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 3 TRAINING CONTRACT"
)

print(
    "============================================================"
)

print(
    "Seed index:                  3 / 3"
)

print(
    "RNG seed:                    252916394"
)

print(
    "Initialization:              FRESH"
)

print(
    "Initial resume:              NONE"
)

print(
    "Full training blocks:        11"
)

print(
    "Microsteps/rank/block:       1000"
)

print(
    "Gradient accumulation:       4"
)

print(
    "Possible updates/block:      250"
)

print(
    "Initial update attempts:     2750"
)

print(
    "Final SUCCESSFUL updates:    EXACTLY 2750"
)

print(
    "Native tile:                 1024"
)

print(
    "GPUs:                        2 x T4"
)

print(
    "Synthetic batch/GPU:         1"
)

print(
    "Anchor pair batch/GPU:       2"
)

print(
    "Effective synthetic batch:   8"
)

print(
    "Synthetic workers/GPU:       2"
)

print(
    "Anchor workers/GPU:          1"
)

print(
    "Learning rate:               3e-4"
)

print(
    "Warmup successful updates:   50"
)

print(
    "Post-warmup LR:              CONSTANT"
)

print(
    "AMP:                         FP16"
)

print(
    "AMP initial scale:           32"
)

print(
    "Original PCC:                RETAINED"
)

print(
    "CRPA:                        RETAINED"
)

print(
    "PCC-center:                  DISABLED"
)

print(
    "Early stopping:              NO"
)

print(
    "Final checkpoint policy:     exact step 2750"
)

print(
    "best.pt final reporting:     NO"
)

print(
    "External data tuning:        NONE"
)

print()


# ============================================================
# 10. FRESH INITIAL 11-BLOCK TRAINING
# ============================================================

initial_cmd = [
    "torchrun",

    "--standalone",

    "--nproc_per_node=2",

    str(
        LAUNCHER
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--config",
    str(
        CONFIG
    ),

    "--output-root",
    str(
        WORKING
    ),

    "--epochs",
    str(
        INITIAL_EPOCHS
    ),

    "--max-train-steps",
    str(
        INITIAL_MICROSTEPS_PER_EPOCH
    ),

    "--max-val-steps",
    "100",

    "--accum",
    str(
        ACCUM
    ),

    "--per-rank-batch",
    "1",

    "--anchor-batch",
    "2",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "STARTING FINAL SEED 3 — FRESH RUN"
)

print(
    "============================================================"
)

print(
    " ".join(
        initial_cmd
    )
)

print()


overall_start = time.time()

initial_start = time.time()


subprocess.run(
    initial_cmd,
    check=True,
)


initial_elapsed = (
    time.time()
    - initial_start
)


# ============================================================
# 11. PRESERVE INITIAL FULL-RUN METADATA
# ============================================================

for path, label in [
    (
        RUN_ROOT,
        "Seed-3 run directory",
    ),
    (
        SNAPSHOT_DIR,
        "Seed-3 snapshots",
    ),
    (
        RUN_STATUS,
        "Seed-3 run status",
    ),
    (
        RESOLVED_CONFIG,
        "Seed-3 resolved config",
    ),
    (
        HISTORY,
        "Seed-3 history",
    ),
]:

    require(
        path,
        label,
    )


shutil.copy2(
    RUN_STATUS,
    INITIAL_STATUS_COPY,
)

shutil.copy2(
    RESOLVED_CONFIG,
    INITIAL_RESOLVED_COPY,
)

shutil.copy2(
    HISTORY,
    INITIAL_HISTORY_COPY,
)


initial_status = load_json(
    INITIAL_STATUS_COPY
)


initial_records = snapshot_records()


if len(
    initial_records
) != INITIAL_EPOCHS:

    raise RuntimeError(
        "Expected exactly 11 snapshots after initial "
        f"Seed-3 run; found {len(initial_records)}."
    )


latest = initial_records[-1]


initial_terminal_checkpoint = (
    latest[
        "path"
    ]
)

initial_terminal_epoch = int(
    latest[
        "epoch"
    ]
)

initial_successful_updates = int(
    latest[
        "optimizer_step"
    ]
)


if initial_terminal_epoch != 10:

    raise RuntimeError(
        "Expected initial terminal epoch 10; "
        f"found {initial_terminal_epoch}."
    )


if (
    initial_successful_updates
    > TARGET_OPTIMIZER_STEP
):

    raise RuntimeError(
        "CRITICAL: initial run exceeded frozen 2750 target."
    )


initial_inferred_amp_skips = (
    EXPECTED_INITIAL_UPDATE_ATTEMPTS
    - initial_successful_updates
)


if initial_inferred_amp_skips < 0:

    raise RuntimeError(
        "Invalid initial AMP-skip inference."
    )


reported_initial_amp_skips = int(
    initial_status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if (
    reported_initial_amp_skips
    != initial_inferred_amp_skips
):

    raise RuntimeError(
        "Initial Seed-3 AMP-skip accounting mismatch.\n"
        f"Inferred from optimizer step: "
        f"{initial_inferred_amp_skips}\n"
        f"run_status reports:          "
        f"{reported_initial_amp_skips}"
    )


print()
print(
    "============================================================"
)

print(
    "INITIAL FINAL-SEED-3 RUN COMPLETE"
)

print(
    "============================================================"
)

print(
    "Successful optimizer updates:",
    initial_successful_updates,
)

print(
    "AMP skipped updates:",
    initial_inferred_amp_skips,
)

print(
    "Initial terminal epoch:",
    initial_terminal_epoch,
)

print(
    "Initial terminal checkpoint:"
)

print(
    initial_terminal_checkpoint
)

print()


# ============================================================
# 12. EXACT-UPDATE COMPENSATION
#
# If AMP skips occurred:
#
# missing_updates * 4 microsteps
#
# are requested.
#
# If another skip occurs in compensation, repeat only for the
# remaining successful-update deficit.
#
# This does not intentionally exceed 2750.
# ============================================================

current_checkpoint = (
    initial_terminal_checkpoint
)

current_epoch = (
    initial_terminal_epoch
)

current_step = (
    initial_successful_updates
)

completion_iterations = []

MAX_COMPLETION_ITERATIONS = 10


while (
    current_step
    < TARGET_OPTIMIZER_STEP
):

    if (
        len(
            completion_iterations
        )
        >= MAX_COMPLETION_ITERATIONS
    ):

        raise RuntimeError(
            "Too many Seed-3 AMP compensation iterations.\n"
            "STOP and inspect."
        )


    missing_updates = (
        TARGET_OPTIMIZER_STEP
        - current_step
    )


    requested_microsteps = (
        missing_updates
        * ACCUM
    )


    next_epoch = (
        current_epoch
        + 1
    )


    epochs_cli = (
        next_epoch
        + 1
    )


    print()
    print(
        "============================================================"
    )

    print(
        "SEED-3 EXACT-UPDATE COMPENSATION "
        f"ITERATION {len(completion_iterations) + 1}"
    )

    print(
        "============================================================"
    )

    print(
        "Current successful updates:",
        current_step,
    )

    print(
        "Remaining successful updates:",
        missing_updates,
    )

    print(
        "Requested microsteps/rank:",
        requested_microsteps,
    )

    print(
        "Resume checkpoint:"
    )

    print(
        current_checkpoint
    )

    print()


    cmd = [
        "torchrun",

        "--standalone",

        "--nproc_per_node=2",

        str(
            LAUNCHER
        ),

        "--project-root",
        str(
            PROJECT_ROOT
        ),

        "--config",
        str(
            CONFIG
        ),

        "--output-root",
        str(
            WORKING
        ),

        "--epochs",
        str(
            epochs_cli
        ),

        "--max-train-steps",
        str(
            requested_microsteps
        ),

        "--max-val-steps",
        "100",

        "--accum",
        str(
            ACCUM
        ),

        "--per-rank-batch",
        "1",

        "--anchor-batch",
        "2",

        "--num-workers",
        "2",

        "--log-every",
        str(
            max(
                1,
                requested_microsteps,
            )
        ),

        "--resume",
        str(
            current_checkpoint
        ),
    ]


    iteration_start = time.time()


    subprocess.run(
        cmd,
        check=True,
    )


    iteration_elapsed = (
        time.time()
        - iteration_start
    )


    records_now = snapshot_records()


    candidates = [
        record
        for record in records_now
        if int(
            record[
                "epoch"
            ]
        ) == next_epoch
    ]


    if len(
        candidates
    ) != 1:

        raise RuntimeError(
            "Expected exactly one snapshot for compensation "
            f"epoch {next_epoch}; found {len(candidates)}."
        )


    new_record = candidates[0]


    new_checkpoint = (
        new_record[
            "path"
        ]
    )


    new_step = int(
        new_record[
            "optimizer_step"
        ]
    )


    new_epoch = int(
        new_record[
            "epoch"
        ]
    )


    if new_step <= current_step:

        raise RuntimeError(
            "Seed-3 compensation made no successful "
            "optimizer progress."
        )


    if new_step > TARGET_OPTIMIZER_STEP:

        raise RuntimeError(
            "CRITICAL: Seed-3 compensation exceeded "
            "the frozen 2750-update target.\n"
            f"Observed: {new_step}"
        )


    successful_new_updates = (
        new_step
        - current_step
    )


    inferred_additional_skips = (
        missing_updates
        - successful_new_updates
    )


    if inferred_additional_skips < 0:

        raise RuntimeError(
            "Invalid compensation AMP-skip inference."
        )


    lr_values = verify_constant_lr(
        new_checkpoint
    )


    completion_iterations.append(
        {
            "iteration":
                len(
                    completion_iterations
                )
                + 1,

            "start_optimizer_step":
                current_step,

            "requested_successful_updates":
                missing_updates,

            "requested_microsteps_per_rank":
                requested_microsteps,

            "successful_updates_observed":
                successful_new_updates,

            "inferred_amp_skips":
                inferred_additional_skips,

            "end_optimizer_step":
                new_step,

            "epoch":
                new_epoch,

            "checkpoint":
                str(
                    new_checkpoint
                ),

            "checkpoint_sha256":
                sha256_file(
                    new_checkpoint
                ),

            "learning_rates":
                lr_values,

            "wall_time_minutes":
                iteration_elapsed
                / 60.0,
        }
    )


    print()
    print(
        "Compensation result:"
    )

    print(
        "  successful new updates:",
        successful_new_updates,
    )

    print(
        "  additional AMP skips:",
        inferred_additional_skips,
    )

    print(
        "  new optimizer step:",
        new_step,
    )

    print(
        "  learning rate:",
        lr_values,
    )


    current_checkpoint = (
        new_checkpoint
    )

    current_epoch = (
        new_epoch
    )

    current_step = (
        new_step
    )


# ============================================================
# 13. EXACT TARGET VERIFICATION
# ============================================================

if current_step != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Seed 3 did not finish at exactly 2750 "
        "successful optimizer updates."
    )


final_checkpoint = (
    current_checkpoint
)

final_epoch = (
    current_epoch
)

final_sha = sha256_file(
    final_checkpoint
)

final_lrs = verify_constant_lr(
    final_checkpoint
)


additional_amp_skips = sum(
    int(
        item[
            "inferred_amp_skips"
        ]
    )
    for item in completion_iterations
)


total_amp_skips = (
    initial_inferred_amp_skips
    + additional_amp_skips
)


overall_elapsed = (
    time.time()
    - overall_start
)


# ============================================================
# 14. FINAL SNAPSHOT CHAIN AUDIT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "AUDITING FINAL SEED-3 SNAPSHOT CHAIN"
)

print(
    "============================================================"
)


final_records = snapshot_records()


if not final_records:

    raise RuntimeError(
        "Seed-3 snapshot chain is empty."
    )


previous_epoch = -1
previous_step = -1

snapshot_manifest_records = []


for record in final_records:

    epoch = int(
        record[
            "epoch"
        ]
    )

    step = int(
        record[
            "optimizer_step"
        ]
    )


    if epoch <= previous_epoch:

        raise RuntimeError(
            "Seed-3 snapshot epochs are not strictly increasing."
        )


    if step <= previous_step:

        raise RuntimeError(
            "Seed-3 optimizer steps are not strictly increasing."
        )


    if step > TARGET_OPTIMIZER_STEP:

        raise RuntimeError(
            "Found Seed-3 checkpoint beyond frozen target:\n"
            f"{record['path']}"
        )


    snapshot_manifest_records.append(
        {
            "epoch":
                epoch,

            "optimizer_step":
                step,

            "path":
                str(
                    record[
                        "path"
                    ]
                ),

            "sha256":
                record[
                    "sha256"
                ],
        }
    )


    print(
        f"PASS  "
        f"epoch={epoch:02d}  "
        f"optimizer_step={step:04d}  "
        f"sha={record['sha256'][:18]}..."
    )


    previous_epoch = epoch
    previous_step = step


if int(
    final_records[
        -1
    ][
        "optimizer_step"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Last Seed-3 snapshot is not step 2750."
    )


if (
    Path(
        final_records[
            -1
        ][
            "path"
        ]
    )
    != final_checkpoint
):

    raise RuntimeError(
        "Final checkpoint identity mismatch."
    )


# ============================================================
# 15. WRITE EXACT-UPDATE PROVENANCE
# ============================================================

exact_update_record = {
    "experiment":
        EXPERIMENT,

    "seed_index":
        SEED_INDEX,

    "rng_seed":
        RNG_SEED,

    "scientific_change":
        False,

    "frozen_target_successful_optimizer_updates":
        TARGET_OPTIMIZER_STEP,

    "initial_training": {
        "fresh_initialization":
            True,

        "resume":
            "none",

        "epochs":
            INITIAL_EPOCHS,

        "microsteps_per_rank_per_epoch":
            INITIAL_MICROSTEPS_PER_EPOCH,

        "gradient_accumulation":
            ACCUM,

        "possible_optimizer_update_attempts":
            EXPECTED_INITIAL_UPDATE_ATTEMPTS,

        "successful_optimizer_updates":
            initial_successful_updates,

        "inferred_amp_skips":
            initial_inferred_amp_skips,

        "run_status_reported_amp_skips":
            reported_initial_amp_skips,

        "terminal_epoch":
            initial_terminal_epoch,

        "terminal_checkpoint":
            str(
                initial_terminal_checkpoint
            ),

        "terminal_checkpoint_sha256":
            sha256_file(
                initial_terminal_checkpoint
            ),

        "wall_time_minutes":
            initial_elapsed
            / 60.0,
    },

    "compensation_required":
        bool(
            initial_successful_updates
            < TARGET_OPTIMIZER_STEP
        ),

    "compensation_policy":
        (
            "If AMP causes skipped optimizer updates, resume the "
            "trusted terminal checkpoint for only enough bounded "
            "microsteps to reach exactly 2750 successful optimizer "
            "updates. No scientific hyperparameter is changed."
        ),

    "compensation_iterations":
        completion_iterations,

    "additional_amp_skips_during_compensation":
        additional_amp_skips,

    "total_amp_skips_observed":
        total_amp_skips,

    "final_successful_optimizer_updates":
        current_step,

    "final_epoch":
        final_epoch,

    "final_checkpoint":
        str(
            final_checkpoint
        ),

    "final_checkpoint_sha256":
        final_sha,

    "final_learning_rates":
        final_lrs,

    "architecture_changed":
        False,

    "loss_changed":
        False,

    "pcc_changed":
        False,

    "crpa_changed":
        False,

    "pcc_center":
        "rejected_and_disabled",

    "external_data_used":
        False,

    "overall_wall_time_minutes":
        overall_elapsed
        / 60.0,
}


EXACT_UPDATE_MANIFEST.write_text(
    json.dumps(
        exact_update_record,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 16. FINAL TRAINING MANIFEST
# ============================================================

final_manifest = {
    "experiment":
        EXPERIMENT,

    "role":
        "FINAL_INDEPENDENT_SEED_3_OF_3",

    "seed_index":
        SEED_INDEX,

    "rng_seed":
        RNG_SEED,

    "fresh_initialization":
        True,

    "initial_resume":
        "none",

    "frozen_successful_optimizer_updates":
        TARGET_OPTIMIZER_STEP,

    "successful_optimizer_updates":
        current_step,

    "initial_amp_skips":
        initial_inferred_amp_skips,

    "additional_compensation_amp_skips":
        additional_amp_skips,

    "total_amp_skips_observed":
        total_amp_skips,

    "compensation_required":
        bool(
            completion_iterations
        ),

    "terminal_checkpoint_policy":
        "exactly_2750_successful_optimizer_updates",

    "best_pt_used_for_final_reporting":
        False,

    "external_data_used_for_selection":
        False,

    "original_pcc":
        "retained",

    "crpa":
        "retained",

    "pcc_center":
        "rejected_and_disabled",

    "config": {
        "path":
            str(
                CONFIG
            ),

        "sha256":
            actual_config_sha,
    },

    "terminal_checkpoint": {
        "path":
            str(
                final_checkpoint
            ),

        "epoch":
            final_epoch,

        "optimizer_step":
            current_step,

        "sha256":
            final_sha,
    },

    "snapshot_chain":
        snapshot_manifest_records,

    "exact_update_provenance": {
        "path":
            str(
                EXACT_UPDATE_MANIFEST
            ),

        "sha256":
            sha256_file(
                EXACT_UPDATE_MANIFEST
            ),
    },

    "multi_seed_plan": {
        "path":
            str(
                PLAN_JSON
            ),

        "sha256":
            sha256_file(
                PLAN_JSON
            ),
    },

    "completed_seed_references": {
        "seed1_terminal_sha256":
            EXPECTED_SEED1_SHA,

        "seed2_terminal_sha256":
            EXPECTED_SEED2_SHA,
    },

    "overall_wall_time_minutes":
        overall_elapsed
        / 60.0,
}


FINAL_TRAINING_MANIFEST.write_text(
    json.dumps(
        final_manifest,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 3 / 3 TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    "RNG seed:                    ",
    RNG_SEED,
)

print(
    "Fresh initialization:         YES"
)

print(
    "Initial successful updates:   ",
    initial_successful_updates,
)

print(
    "Initial AMP skips:             ",
    initial_inferred_amp_skips,
)

print(
    "Additional AMP skips:          ",
    additional_amp_skips,
)

print(
    "Total AMP skips observed:      ",
    total_amp_skips,
)

print(
    "FINAL successful updates:      ",
    current_step,
)

print(
    "Final checkpoint epoch:        ",
    final_epoch,
)

print(
    "Compensation iterations:       ",
    len(
        completion_iterations
    ),
)

print(
    "Final LR:                      ",
    final_lrs,
)

print(
    "Overall wall time:             "
    f"{overall_elapsed / 60.0:.2f} min"
)

print()

print(
    "FINAL SEED-3 TERMINAL MODEL:"
)

print(
    final_checkpoint
)

print()

print(
    "SHA256:"
)

print(
    final_sha
)

print()

print(
    "Exact-update provenance:"
)

print(
    EXACT_UPDATE_MANIFEST
)

print()

print(
    "Final Seed-3 training manifest:"
)

print(
    FINAL_TRAINING_MANIFEST
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FINAL SEED 3 EXACT 2750 UPDATES: PASS"
)

print(
    "============================================================"
)

print(
    "STOP HERE."
)

print(
    "ALL THREE FINAL TRAINING SEEDS WILL THEN BE COMPLETE."
)

print(
    "Next: fixed 1000-scene Seed-3 QC."
)

DASO-Net — FINAL SEED 3 / 3 PREFLIGHT
PASS  Frozen project dataset
PASS  Restored validated code tree
PASS  Validated trainer
PASS  Validated checkpoint launcher
PASS  Validated decoder
PASS  Frozen multi-seed plan
PASS  Frozen Seed-3 config
PASS  After-Seed2 restore audit
PASS  Final Seed-1 terminal checkpoint
PASS  Final Seed-2 terminal checkpoint

After-Seed2 restore state: PASS
Seed-1 terminal remains intact: PASS
Seed-2 terminal remains intact: PASS
Validated decoder SHA: PASS
Frozen multi-seed plan: PASS
Seed-3 config SHA: PASS
Seed-3 RNG seed: 252916394 — PASS
PCC-center disabled: PASS

Fresh initialization condition: PASS
Seed-3 output directory is absent.
GPU runtime: 2x Tesla T4 — PASS

FINAL SEED 3 TRAINING CONTRACT
Seed index:                  3 / 3
RNG seed:                    252916394
Initialization:              FRESH
Initial resume:              NONE
Full training blocks:        11
Microsteps/rank/block:       1000
Gradient accumulation:       4
Possible updates/block:

W0918 10:10:53.826000 122 torch/distributed/run.py:852] 
W0918 10:10:53.826000 122 torch/distributed/run.py:852] *****************************************
W0918 10:10:53.826000 122 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0918 10:10:53.826000 122 torch/distributed/run.py:852] *****************************************
[W918 10:10:54.115063045 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W918 10:10:56.823543319 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W918 10:10:56.826843935 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_final_seed03_s252916394
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      2750
Smoke mode:                 False
epoch=00 step=00000/01000 loss=8.7849 inst=8.0868 pcc=0.0002 crpa=0.6979 lr=6.00e-06 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00100/01000 loss=7.9816 inst=7.3945 pcc=0.0000 crpa=0.5870 lr=1.56e-04 vram_alloc=8.97GB vram_res=9.11GB
epoch=00 step=00200/01000 loss=6.0499 inst=5.4760 pcc=0.0000 crpa=0.5739 l

In [4]:
# ============================================================
# DASO-Net — FINAL SEED 3 FIXED 1000-SCENE QC
#
# Seed 3:
#   RNG seed: 252916394
#   terminal: exactly 2750 successful optimizer updates
#
# Frozen QC protocol:
#   split: val
#   scenes: 1000
#   evaluation seed: 20260917
#   decode min score: 0.05
#   operating score: 0.05
#   top-k: 100
#   mask threshold: 0.5
#
# Runs:
#   A. Instance metrics
#   B. Oracle component diagnostic
#   C. Counterfactual diagnostic
#
# NO TRAINING.
# NO EXTERNAL DATA.
# NO MODEL SELECTION.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path


# ============================================================
# 1. FROZEN CONSTANTS
# ============================================================

SEED_INDEX = 3
TRAINING_SEED = 252916394

VAL_SCENES = 1000
VAL_SEED = 20260917

DECODE_MIN_SCORE = 0.05
OPERATING_SCORE = 0.05
TOPK = 100
MASK_THRESHOLD = 0.5
PEAK_KERNEL = 3

EXPECTED_UPDATES = 2750

EXPECTED_TERMINAL_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0eb9018ec9133486512bd1f87153f447"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# 2. PATHS
# ============================================================

WORKING = Path("/kaggle/working")

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

RUN_ROOT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
)

TERMINAL = (
    RUN_ROOT
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

TRAINING_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED03_TRAINING_MANIFEST.json"
)

EXACT_UPDATE_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED03_EXACT_UPDATE_COMPLETION.json"
)

QC_ROOT = (
    WORKING
    / "dasonet_final_qc"
    / "seed03_s252916394_val1000_v1_fixedop005"
)

METRICS_DIR = QC_ROOT / "metrics"
ORACLE_DIR = QC_ROOT / "oracle"
CF_DIR = QC_ROOT / "counterfactual"

METRIC_JSON = (
    METRICS_DIR
    / "phase5c_metric_summary.json"
)

PER_SCENE_CSV = (
    METRICS_DIR
    / "phase5c_per_scene.csv"
)

ORACLE_JSON = (
    ORACLE_DIR
    / "oracle_component_diagnostic.json"
)

ORACLE_CSV = (
    ORACLE_DIR
    / "oracle_particle_diagnostic.csv"
)

CF_JSON = (
    CF_DIR
    / "counterfactual_center_diagnostic.json"
)

QC_SETTINGS = (
    QC_ROOT
    / "QC_SETTINGS.json"
)

QC_MANIFEST = (
    QC_ROOT
    / "FINAL_SEED03_QC_RUN_MANIFEST.json"
)

SEED1_PER_SCENE = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
    / "metrics"
    / "phase5c_per_scene.csv"
)


# ============================================================
# 3. HELPERS
# ============================================================

def require(path: Path, label: str) -> None:
    if not path.exists():
        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def load_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(path: Path, obj) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def script_help(path: Path) -> str:
    result = subprocess.run(
        [
            sys.executable,
            str(path),
            "--help",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Could not read CLI help:\n{path}\n\n"
            f"{result.stdout}"
        )

    return result.stdout


def run_command(
    title: str,
    command: list[str],
    expected_output: Path,
) -> float:

    print()
    print("=" * 60)
    print(title)
    print("=" * 60)
    print("Command:")
    print(" ".join(command))
    print()

    env = os.environ.copy()

    # Fixed single-GPU QC.
    env["CUDA_VISIBLE_DEVICES"] = "0"

    start = time.time()

    subprocess.run(
        command,
        check=True,
        env=env,
    )

    elapsed = time.time() - start

    require(
        expected_output,
        f"{title} output",
    )

    print()
    print(f"{title}: PASS")
    print(f"Runtime: {elapsed / 60.0:.2f} min")

    return elapsed


def read_csv(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(csv.DictReader(f))


def find_scene_column(rows):
    if not rows:
        return None

    lookup = {
        str(k).lower(): k
        for k in rows[0].keys()
        if k is not None
    }

    for candidate in [
        "scene_id",
        "scene",
        "recipe_id",
        "sample_id",
        "id",
    ]:
        if candidate in lookup:
            return lookup[candidate]

    return None


# ============================================================
# 4. PREFLIGHT
# ============================================================

print("=" * 60)
print("DASO-Net — FINAL SEED 3 FIXED 1000-SCENE QC")
print("=" * 60)

for path, label in [
    (PROJECT_ROOT, "Frozen project dataset"),
    (EVALUATOR, "Phase5C evaluator"),
    (ORACLE, "Oracle diagnostic"),
    (COUNTERFACTUAL, "Counterfactual diagnostic"),
    (DECODER, "Validated decoder"),
    (TERMINAL, "Seed-3 terminal checkpoint"),
    (TRAINING_MANIFEST, "Seed-3 training manifest"),
    (EXACT_UPDATE_MANIFEST, "Seed-3 exact-update manifest"),
    (SEED1_PER_SCENE, "Seed-1 fixed-QC scene reference"),
]:
    require(path, label)
    print(f"PASS  {label}")


# ============================================================
# 5. MODEL / TRAINING INTEGRITY
# ============================================================

terminal_sha = sha256_file(
    TERMINAL
)

if terminal_sha != EXPECTED_TERMINAL_SHA:
    raise RuntimeError(
        "Seed-3 terminal SHA mismatch.\n"
        f"Expected: {EXPECTED_TERMINAL_SHA}\n"
        f"Actual:   {terminal_sha}"
    )


if sha256_file(DECODER) != EXPECTED_DECODER_SHA:
    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


training_manifest = load_json(
    TRAINING_MANIFEST
)

exact_manifest = load_json(
    EXACT_UPDATE_MANIFEST
)


if int(
    training_manifest["rng_seed"]
) != TRAINING_SEED:
    raise RuntimeError(
        "Seed-3 RNG identity mismatch."
    )


if int(
    training_manifest[
        "successful_optimizer_updates"
    ]
) != EXPECTED_UPDATES:
    raise RuntimeError(
        "Seed-3 training manifest is not at 2750."
    )


if int(
    exact_manifest[
        "final_successful_optimizer_updates"
    ]
) != EXPECTED_UPDATES:
    raise RuntimeError(
        "Seed-3 exact-update manifest is not at 2750."
    )


print()
print("Seed-3 terminal SHA: PASS")
print("Seed-3 RNG 252916394: PASS")
print("Seed-3 exact 2750 updates: PASS")
print("Validated decoder SHA: PASS")


# ============================================================
# 6. CLI AUDIT
# ============================================================

eval_help = script_help(
    EVALUATOR
)

oracle_help = script_help(
    ORACLE
)

cf_help = script_help(
    COUNTERFACTUAL
)


for flag in [
    "--project-root",
    "--checkpoint",
    "--output-dir",
    "--split",
    "--max-scenes",
    "--seed",
    "--decode-min-score",
    "--operating-score",
    "--topk",
    "--mask-threshold",
]:
    if flag not in eval_help:
        raise RuntimeError(
            f"Evaluator missing expected flag: {flag}"
        )


if "--evaluator" not in oracle_help:
    raise RuntimeError(
        "Oracle script does not expose required --evaluator."
    )


print()
print("Evaluator CLI contract: PASS")
print("Oracle --evaluator: PASS")
print(
    "Counterfactual --evaluator:",
    "YES"
    if "--evaluator" in cf_help
    else "NOT REQUIRED",
)


# ============================================================
# 7. CREATE FRESH QC ROOT
# ============================================================

if QC_ROOT.exists():
    raise RuntimeError(
        "Seed-3 QC directory already exists:\n"
        f"{QC_ROOT}\n\n"
        "STOP rather than overwrite or mix results."
    )


METRICS_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

ORACLE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

CF_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


settings = {
    "qc_name":
        "DASO-Net Final Seed-3 fixed 1000-scene QC",

    "seed_index":
        SEED_INDEX,

    "training_rng_seed":
        TRAINING_SEED,

    "terminal_checkpoint":
        str(TERMINAL),

    "terminal_sha256":
        terminal_sha,

    "successful_optimizer_updates":
        EXPECTED_UPDATES,

    "split":
        "val",

    "num_scenes":
        VAL_SCENES,

    "evaluation_seed":
        VAL_SEED,

    "decode_min_score":
        DECODE_MIN_SCORE,

    "operating_score":
        OPERATING_SCORE,

    "topk":
        TOPK,

    "mask_threshold":
        MASK_THRESHOLD,

    "oracle_peak_kernel":
        PEAK_KERNEL,

    "external_data_used":
        False,

    "model_selection":
        "none",
}


save_json(
    QC_SETTINGS,
    settings,
)


print()
print("QC settings frozen: PASS")
print("Scenes:              1000")
print("Evaluation seed:     20260917")
print("Decode min score:    0.05")
print("Operating score:     0.05")
print("Top-k:               100")
print("Mask threshold:      0.5")
print("External data:       NONE")
print("Model selection:     NONE")


# ============================================================
# 8. STAGE A — MAIN METRICS
# ============================================================

metrics_cmd = [
    sys.executable,
    str(EVALUATOR),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(TERMINAL),

    "--output-dir",
    str(METRICS_DIR),

    "--split",
    "val",

    "--max-scenes",
    str(VAL_SCENES),

    "--seed",
    str(VAL_SEED),

    "--decode-min-score",
    str(DECODE_MIN_SCORE),

    "--operating-score",
    str(OPERATING_SCORE),

    "--topk",
    str(TOPK),

    "--mask-threshold",
    str(MASK_THRESHOLD),
]


t_metrics = run_command(
    "STAGE A — 1000-SCENE METRICS",
    metrics_cmd,
    METRIC_JSON,
)


require(
    PER_SCENE_CSV,
    "Seed-3 per-scene metric CSV",
)


seed3_rows = read_csv(
    PER_SCENE_CSV
)


if len(seed3_rows) != 1000:
    raise RuntimeError(
        "Seed-3 metric CSV does not contain "
        f"exactly 1000 scenes: {len(seed3_rows)}"
    )


print()
print("Seed-3 metric scene count: 1000 — PASS")


# ============================================================
# 9. SAME ORDERED SCENE SET AS SEED 1
# ============================================================

seed1_rows = read_csv(
    SEED1_PER_SCENE
)


if len(seed1_rows) != 1000:
    raise RuntimeError(
        "Seed-1 QC reference does not contain 1000 scenes."
    )


col1 = find_scene_column(
    seed1_rows
)

col3 = find_scene_column(
    seed3_rows
)


if col1 is None or col3 is None:
    raise RuntimeError(
        "Could not identify scene-ID columns for "
        "cross-seed scene verification."
    )


seed1_ids = [
    str(row[col1])
    for row in seed1_rows
]

seed3_ids = [
    str(row[col3])
    for row in seed3_rows
]


if seed1_ids != seed3_ids:
    raise RuntimeError(
        "Seed-3 QC scene set/order differs from Seed 1."
    )


print(
    "Seed-1 / Seed-3 ordered 1000-scene IDs identical: PASS"
)


# ============================================================
# 10. STAGE B — ORACLE DIAGNOSTIC
# ============================================================

oracle_cmd = [
    sys.executable,
    str(ORACLE),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(TERMINAL),

    "--evaluator",
    str(EVALUATOR),

    "--output-dir",
    str(ORACLE_DIR),

    "--split",
    "val",

    "--max-scenes",
    str(VAL_SCENES),

    "--seed",
    str(VAL_SEED),

    "--peak-kernel",
    str(PEAK_KERNEL),

    "--topk",
    str(TOPK),

    "--mask-threshold",
    str(MASK_THRESHOLD),
]


t_oracle = run_command(
    "STAGE B — 1000-SCENE ORACLE DIAGNOSTIC",
    oracle_cmd,
    ORACLE_JSON,
)


require(
    ORACLE_CSV,
    "Seed-3 oracle particle CSV",
)


# ============================================================
# 11. STAGE C — COUNTERFACTUAL DIAGNOSTIC
# ============================================================

cf_cmd = [
    sys.executable,
    str(COUNTERFACTUAL),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(TERMINAL),

    "--output-dir",
    str(CF_DIR),

    "--split",
    "val",

    "--max-scenes",
    str(VAL_SCENES),

    "--seed",
    str(VAL_SEED),
]


# Add evaluator only if this restored CLI exposes it.
if "--evaluator" in cf_help:
    cf_cmd.extend(
        [
            "--evaluator",
            str(EVALUATOR),
        ]
    )


if "--peak-kernel" in cf_help:
    cf_cmd.extend(
        [
            "--peak-kernel",
            str(PEAK_KERNEL),
        ]
    )


if "--topk" in cf_help:
    cf_cmd.extend(
        [
            "--topk",
            str(TOPK),
        ]
    )


if "--mask-threshold" in cf_help:
    cf_cmd.extend(
        [
            "--mask-threshold",
            str(MASK_THRESHOLD),
        ]
    )


t_cf = run_command(
    "STAGE C — 1000-SCENE COUNTERFACTUAL DIAGNOSTIC",
    cf_cmd,
    CF_JSON,
)


# ============================================================
# 12. FINAL OUTPUT HASH AUDIT
# ============================================================

outputs = {
    "metric_summary":
        METRIC_JSON,

    "per_scene_csv":
        PER_SCENE_CSV,

    "oracle_json":
        ORACLE_JSON,

    "oracle_csv":
        ORACLE_CSV,

    "counterfactual_json":
        CF_JSON,

    "qc_settings":
        QC_SETTINGS,
}


output_hashes = {}


for name, path in outputs.items():

    require(
        path,
        name,
    )

    output_hashes[name] = sha256_file(
        path
    )


# ============================================================
# 13. WRITE NON-FRAGILE QC MANIFEST
#
# The scientific JSON outputs remain canonical.
# We intentionally do NOT use generic metric-key extraction
# here. The script stdout and source JSONs are the evidence.
# ============================================================

manifest = {
    "qc_name":
        "DASO-Net Final Seed-3 fixed 1000-scene QC",

    "status":
        "STAGES_COMPLETE",

    "seed_index":
        3,

    "training_rng_seed":
        TRAINING_SEED,

    "terminal_checkpoint": {
        "path":
            str(TERMINAL),

        "sha256":
            terminal_sha,

        "successful_optimizer_updates":
            EXPECTED_UPDATES,
    },

    "evaluation_protocol": {
        "split":
            "val",

        "num_scenes":
            1000,

        "evaluation_seed":
            VAL_SEED,

        "decode_min_score":
            DECODE_MIN_SCORE,

        "operating_score":
            OPERATING_SCORE,

        "topk":
            TOPK,

        "mask_threshold":
            MASK_THRESHOLD,

        "same_ordered_scene_set_as_seed1":
            True,

        "external_data_used":
            False,

        "model_selection":
            "none",
    },

    "canonical_outputs": {
        name:
            str(path)
        for name, path in outputs.items()
    },

    "canonical_output_sha256":
        output_hashes,

    "runtime_minutes": {
        "metrics":
            t_metrics / 60.0,

        "oracle":
            t_oracle / 60.0,

        "counterfactual":
            t_cf / 60.0,

        "total":
            (
                t_metrics
                + t_oracle
                + t_cf
            )
            / 60.0,
    },

    "training_performed":
        False,

    "scientific_role":
        (
            "Final-seed synthetic-validation integrity/sanity QC. "
            "Not used to select, discard, or rank final seeds."
        ),
}


save_json(
    QC_MANIFEST,
    manifest,
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print()
print("=" * 60)
print("FINAL SEED 3 — FIXED QC STAGES COMPLETE")
print("=" * 60)

print()
print("Seed index:                 3 / 3")
print("Training RNG seed:          252916394")
print("Successful updates:         2750")

print()
print("Terminal SHA256:")
print(terminal_sha)

print()
print("Validation scenes:          1000")
print("Evaluation seed:            20260917")
print("Decode minimum score:       0.05")
print("Operating score:            0.05")
print("Same scene set as Seed 1:   YES")
print("External data:              NONE")
print("Model selection:            NONE")

print()
print("Canonical metric JSON:")
print(METRIC_JSON)

print()
print("Canonical oracle JSON:")
print(ORACLE_JSON)

print()
print("Canonical counterfactual JSON:")
print(CF_JSON)

print()
print("QC manifest:")
print(QC_MANIFEST)

print()
print(
    f"Total QC runtime: "
    f"{(t_metrics + t_oracle + t_cf) / 60.0:.2f} min"
)

print()
print("=" * 60)
print("DASO-NET FINAL SEED 3 FIXED QC STAGES: PASS")
print("=" * 60)

print("NO TRAINING WAS PERFORMED.")
print("DO NOT START ANY MORE TRAINING.")
print(
    "Paste the METRIC, ORACLE, COUNTERFACTUAL summaries "
    "and this final PASS section."
)

DASO-Net — FINAL SEED 3 FIXED 1000-SCENE QC
PASS  Frozen project dataset
PASS  Phase5C evaluator
PASS  Oracle diagnostic
PASS  Counterfactual diagnostic
PASS  Validated decoder
PASS  Seed-3 terminal checkpoint
PASS  Seed-3 training manifest
PASS  Seed-3 exact-update manifest
PASS  Seed-1 fixed-QC scene reference

Seed-3 terminal SHA: PASS
Seed-3 RNG 252916394: PASS
Seed-3 exact 2750 updates: PASS
Validated decoder SHA: PASS

Evaluator CLI contract: PASS
Oracle --evaluator: PASS
Counterfactual --evaluator: YES

QC settings frozen: PASS
Scenes:              1000
Evaluation seed:     20260917
Decode min score:    0.05
Operating score:     0.05
Top-k:               100
Mask threshold:      0.5
External data:       NONE
Model selection:     NONE

STAGE A — 1000-SCENE METRICS
Command:
/usr/bin/python3 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --checkpoint /kaggle/working/das

In [5]:
# ============================================================
# DASO-Net — FINALIZE SEED-3 QC
#
# DOES NOT rerun:
#   - training
#   - metric evaluation
#   - oracle diagnostic
#   - counterfactual diagnostic
#
# It verifies the already-created canonical outputs and writes:
#
#   FINAL_SEED03_QC_SUMMARY.json
#   FINAL_SEED03_QC_FINALIZATION_MANIFEST.json
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
from pathlib import Path


# ============================================================
# 1. CONSTANTS
# ============================================================

TRAINING_SEED = 252916394

EXPECTED_UPDATES = 2750

EXPECTED_TERMINAL_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0eb9018ec9133486512bd1f87153f447"
)

EVAL_SEED = 20260917


# ============================================================
# 2. PATHS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

RUN_ROOT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
)

TERMINAL = (
    RUN_ROOT
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

TRAINING_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED03_TRAINING_MANIFEST.json"
)

EXACT_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED03_EXACT_UPDATE_COMPLETION.json"
)

QC_ROOT = (
    WORKING
    / "dasonet_final_qc"
    / "seed03_s252916394_val1000_v1_fixedop005"
)

METRIC_JSON = (
    QC_ROOT
    / "metrics"
    / "phase5c_metric_summary.json"
)

PER_SCENE_CSV = (
    QC_ROOT
    / "metrics"
    / "phase5c_per_scene.csv"
)

ORACLE_JSON = (
    QC_ROOT
    / "oracle"
    / "oracle_component_diagnostic.json"
)

ORACLE_CSV = (
    QC_ROOT
    / "oracle"
    / "oracle_particle_diagnostic.csv"
)

CF_JSON = (
    QC_ROOT
    / "counterfactual"
    / "counterfactual_center_diagnostic.json"
)

QC_SETTINGS = (
    QC_ROOT
    / "QC_SETTINGS.json"
)

STAGE_MANIFEST = (
    QC_ROOT
    / "FINAL_SEED03_QC_RUN_MANIFEST.json"
)

FINAL_SUMMARY = (
    QC_ROOT
    / "FINAL_SEED03_QC_SUMMARY.json"
)

FINALIZATION_MANIFEST = (
    QC_ROOT
    / "FINAL_SEED03_QC_FINALIZATION_MANIFEST.json"
)

SEED1_PER_SCENE = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
    / "metrics"
    / "phase5c_per_scene.csv"
)


# ============================================================
# 3. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def collect_numbers(
    obj,
):
    out = []

    if isinstance(
        obj,
        bool,
    ):
        return out

    if isinstance(
        obj,
        (int, float),
    ):

        value = float(
            obj
        )

        if math.isfinite(
            value
        ):
            out.append(
                value
            )

        return out

    if isinstance(
        obj,
        dict,
    ):

        for value in obj.values():

            out.extend(
                collect_numbers(
                    value
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for value in obj:

            out.extend(
                collect_numbers(
                    value
                )
            )

    return out


def contains_close(
    values,
    target,
    tolerance=5e-4,
) -> bool:

    return any(
        abs(
            float(value)
            - float(target)
        )
        <= tolerance
        for value in values
    )


def read_csv(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def scene_column(
    rows,
):

    if not rows:
        return None

    lookup = {
        str(k).lower():
            k
        for k in rows[0].keys()
        if k is not None
    }

    for candidate in [
        "scene_id",
        "scene",
        "recipe_id",
        "sample_id",
        "id",
    ]:

        if candidate in lookup:

            return lookup[
                candidate
            ]

    return None


# ============================================================
# 4. REQUIRE FILES
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINALIZE SEED-3 QC"
)

print(
    "============================================================"
)


for path, label in [
    (
        TERMINAL,
        "Seed-3 terminal checkpoint",
    ),
    (
        TRAINING_MANIFEST,
        "Seed-3 training manifest",
    ),
    (
        EXACT_MANIFEST,
        "Seed-3 exact-update manifest",
    ),
    (
        QC_SETTINGS,
        "Seed-3 QC settings",
    ),
    (
        STAGE_MANIFEST,
        "Seed-3 QC stage manifest",
    ),
    (
        METRIC_JSON,
        "Metric summary JSON",
    ),
    (
        PER_SCENE_CSV,
        "Per-scene CSV",
    ),
    (
        ORACLE_JSON,
        "Oracle JSON",
    ),
    (
        ORACLE_CSV,
        "Oracle CSV",
    ),
    (
        CF_JSON,
        "Counterfactual JSON",
    ),
    (
        SEED1_PER_SCENE,
        "Seed-1 scene reference",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. TRAINING IDENTITY
# ============================================================

terminal_sha = sha256_file(
    TERMINAL
)


if terminal_sha != EXPECTED_TERMINAL_SHA:

    raise RuntimeError(
        "Seed-3 terminal SHA mismatch."
    )


training = load_json(
    TRAINING_MANIFEST
)

exact = load_json(
    EXACT_MANIFEST
)


if int(
    training[
        "rng_seed"
    ]
) != TRAINING_SEED:

    raise RuntimeError(
        "Seed-3 RNG mismatch."
    )


if int(
    training[
        "successful_optimizer_updates"
    ]
) != EXPECTED_UPDATES:

    raise RuntimeError(
        "Seed-3 training manifest is not at 2750."
    )


if int(
    exact[
        "final_successful_optimizer_updates"
    ]
) != EXPECTED_UPDATES:

    raise RuntimeError(
        "Seed-3 exact-update manifest is not at 2750."
    )


print()
print(
    "Seed-3 terminal SHA: PASS"
)

print(
    "Seed-3 RNG identity: PASS"
)

print(
    "Seed-3 exact 2750 updates: PASS"
)


# ============================================================
# 6. SAME 1000-SCENE SET
# ============================================================

seed1_rows = read_csv(
    SEED1_PER_SCENE
)

seed3_rows = read_csv(
    PER_SCENE_CSV
)


if len(
    seed1_rows
) != 1000:

    raise RuntimeError(
        "Seed-1 reference is not 1000 scenes."
    )


if len(
    seed3_rows
) != 1000:

    raise RuntimeError(
        "Seed-3 QC is not 1000 scenes."
    )


c1 = scene_column(
    seed1_rows
)

c3 = scene_column(
    seed3_rows
)


if c1 is None or c3 is None:

    raise RuntimeError(
        "Could not identify scene ID columns."
    )


ids1 = [
    str(
        row[
            c1
        ]
    )
    for row in seed1_rows
]

ids3 = [
    str(
        row[
            c3
        ]
    )
    for row in seed3_rows
]


if ids1 != ids3:

    raise RuntimeError(
        "Seed-1 and Seed-3 ordered scene sets differ."
    )


print(
    "Same ordered 1000-scene set as Seed 1: PASS"
)


# ============================================================
# 7. LOAD CANONICAL OUTPUTS
# ============================================================

metric_obj = load_json(
    METRIC_JSON
)

oracle_obj = load_json(
    ORACLE_JSON
)

cf_obj = load_json(
    CF_JSON
)


metric_numbers = collect_numbers(
    metric_obj
)

oracle_numbers = collect_numbers(
    oracle_obj
)

cf_numbers = collect_numbers(
    cf_obj
)


# ============================================================
# 8. HEADLINE VALUES PRINTED BY THE CANONICAL SCRIPTS
# ============================================================

metrics = {
    "ap50":
        0.2591,

    "ap75":
        0.0547,

    "ap50_95":
        0.0966,

    "operating_precision_at_0_05":
        0.0181,

    "operating_recall_at_0_05":
        0.3859,

    "operating_f1_at_0_05":
        0.0346,

    "matched_mask_dice":
        0.8096,

    "count_mae_at_0_05":
        43.5730,

    "empty_scene_fp_per_image_at_0_05":
        44.8725,

    "best_f1_threshold":
        0.35,

    "best_f1":
        0.3953,
}


oracle = {
    "gt_score_mean":
        0.2663,

    "gt_score_median":
        0.1655,

    "peak_recall_r4":
        0.5251,

    "peak_recall_r8":
        0.7751,

    "peak_recall_r16":
        0.9004,

    "peak_recall_r32":
        0.9437,

    "center_error_mean_px":
        1.327,

    "box_iou_mean":
        0.5793,

    "box_iou_ge_0_50":
        0.6383,

    "box_iou_ge_0_75":
        0.2817,

    "width_ratio_median":
        0.9765,

    "height_ratio_median":
        0.9990,

    "raw_mask_iou_mean":
        0.1158,

    "raw_mask_iou_ge_0_50":
        0.0503,

    "gt_box_crop_mask_iou_mean":
        0.4712,

    "gt_box_crop_mask_iou_ge_0_50":
        0.4874,

    "pred_box_crop_mask_iou_mean":
        0.4169,

    "pred_box_crop_mask_iou_ge_0_50":
        0.4008,

    "empty_max_peak_score_mean":
        0.2216,

    "empty_max_peak_score_median":
        0.2015,
}


counterfactual = {
    "exact_clean_mean":
        0.0111,

    "exact_composite_mean":
        0.2663,

    "exact_lift_mean":
        0.2552,

    "exact_lift_median":
        0.1564,

    "exact_composite_gt_clean":
        0.9744,

    "exact_lift_ge_0_01":
        0.8669,

    "exact_lift_ge_0_05":
        0.6927,

    "local_clean_mean":
        0.0161,

    "local_composite_mean":
        0.3455,

    "local_lift_mean":
        0.3294,

    "local_lift_median":
        0.3295,

    "local_composite_gt_clean":
        0.9115,

    "local_lift_ge_0_01":
        0.8524,

    "local_lift_ge_0_05":
        0.7495,

    "clean_max_peak_mean":
        0.2284,

    "outside_effect_max_peak_mean":
        0.2140,

    "effect_region_max_peak_mean":
        0.4248,
}


# ============================================================
# 9. VERIFY VALUES ARE ACTUALLY PRESENT IN SOURCE JSONS
# ============================================================

for key, value in metrics.items():

    if not contains_close(
        metric_numbers,
        value,
    ):

        raise RuntimeError(
            f"Metric value not verified in source JSON: "
            f"{key}={value}"
        )


for key, value in oracle.items():

    if not contains_close(
        oracle_numbers,
        value,
    ):

        raise RuntimeError(
            f"Oracle value not verified in source JSON: "
            f"{key}={value}"
        )


for key, value in counterfactual.items():

    if not contains_close(
        cf_numbers,
        value,
    ):

        raise RuntimeError(
            "Counterfactual value not verified in source JSON: "
            f"{key}={value}"
        )


print()
print(
    "Headline values verified against canonical JSONs: PASS"
)


# ============================================================
# 10. SCIENTIFIC SANITY
# ============================================================

sanity_flags = {
    "terminal_sha_verified":
        True,

    "exact_2750_successful_updates":
        True,

    "same_ordered_1000_scenes_as_seed1":
        True,

    "ap50_valid":
        0.0
        <= metrics[
            "ap50"
        ]
        <= 1.0,

    "ap50_95_valid":
        0.0
        <= metrics[
            "ap50_95"
        ]
        <= 1.0,

    "exact_counterfactual_lift_positive":
        counterfactual[
            "exact_lift_mean"
        ]
        > 0.0,

    "local_counterfactual_lift_positive":
        counterfactual[
            "local_lift_mean"
        ]
        > 0.0,

    "effect_response_exceeds_outside_effect":
        counterfactual[
            "effect_region_max_peak_mean"
        ]
        >
        counterfactual[
            "outside_effect_max_peak_mean"
        ],

    "composite_exact_above_clean_majority":
        counterfactual[
            "exact_composite_gt_clean"
        ]
        > 0.5,

    "composite_local_above_clean_majority":
        counterfactual[
            "local_composite_gt_clean"
        ]
        > 0.5,
}


if not all(
    sanity_flags.values()
):

    failed = [
        key
        for key, value
        in sanity_flags.items()
        if not value
    ]

    raise RuntimeError(
        "Seed-3 QC sanity failure:\n"
        + "\n".join(
            failed
        )
    )


print(
    "All Seed-3 QC sanity flags: PASS"
)


# ============================================================
# 11. SOURCE HASHES
# ============================================================

source_hashes = {
    "metric_summary":
        sha256_file(
            METRIC_JSON
        ),

    "per_scene_csv":
        sha256_file(
            PER_SCENE_CSV
        ),

    "oracle_json":
        sha256_file(
            ORACLE_JSON
        ),

    "oracle_csv":
        sha256_file(
            ORACLE_CSV
        ),

    "counterfactual_json":
        sha256_file(
            CF_JSON
        ),

    "qc_settings":
        sha256_file(
            QC_SETTINGS
        ),

    "qc_stage_manifest":
        sha256_file(
            STAGE_MANIFEST
        ),
}


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

summary = {
    "qc_name":
        "DASO-Net Final Seed-3 fixed synthetic-validation QC",

    "status":
        "PASS",

    "seed_index":
        3,

    "training_rng_seed":
        TRAINING_SEED,

    "terminal_checkpoint": {
        "path":
            str(
                TERMINAL
            ),

        "sha256":
            terminal_sha,

        "successful_optimizer_updates":
            EXPECTED_UPDATES,
    },

    "evaluation_protocol": {
        "split":
            "val",

        "num_scenes":
            1000,

        "evaluation_seed":
            EVAL_SEED,

        "decode_min_score":
            0.05,

        "operating_score":
            0.05,

        "topk":
            100,

        "mask_threshold":
            0.5,

        "same_ordered_scene_set_as_seed1":
            True,

        "external_data_used":
            False,

        "model_selection":
            "none",
    },

    "metrics":
        metrics,

    "oracle":
        oracle,

    "counterfactual":
        counterfactual,

    "sanity_flags":
        sanity_flags,

    "canonical_source_sha256":
        source_hashes,

    "scientific_role":
        (
            "Final-seed synthetic-validation integrity/sanity QC. "
            "Not used to select, reject, or rank final seeds."
        ),
}


save_json(
    FINAL_SUMMARY,
    summary,
)


finalization = {
    "status":
        "PASS",

    "seed_index":
        3,

    "training_rng_seed":
        TRAINING_SEED,

    "successful_optimizer_updates":
        EXPECTED_UPDATES,

    "terminal_sha256":
        terminal_sha,

    "final_qc_summary":
        str(
            FINAL_SUMMARY
        ),

    "final_qc_summary_sha256":
        sha256_file(
            FINAL_SUMMARY
        ),

    "source_sha256":
        source_hashes,

    "training_performed_by_finalizer":
        False,

    "inference_performed_by_finalizer":
        False,

    "external_data_used":
        False,

    "model_selection_performed":
        False,
}


save_json(
    FINALIZATION_MANIFEST,
    finalization,
)


# ============================================================
# 13. REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 3 QC FINALIZATION COMPLETE"
)

print(
    "============================================================"
)

print(
    "AP50:                    0.2591"
)

print(
    "AP75:                    0.0547"
)

print(
    "AP50:95:                 0.0966"
)

print(
    "BestF1:                  0.3953"
)

print(
    "Best threshold:          0.35"
)

print()

print(
    "Oracle GT score:         0.2663"
)

print(
    "Oracle PeakRecall@8:     0.7751"
)

print(
    "Oracle BoxIoU:           0.5793"
)

print(
    "Oracle GT-box MaskIoU:   0.4712"
)

print()

print(
    "CF exact lift:          +0.2552"
)

print(
    "CF local lift:          +0.3294"
)

print(
    "CF comp > clean exact:   0.9744"
)

print(
    "CF effect max:           0.4248"
)

print(
    "CF outside max:          0.2140"
)

print()

print(
    "Final summary:"
)

print(
    FINAL_SUMMARY
)

print()

print(
    "Summary SHA256:"
)

print(
    sha256_file(
        FINAL_SUMMARY
    )
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FINAL SEED 3 QC: PASS"
)

print(
    "============================================================"
)

print(
    "NO TRAINING OR INFERENCE WAS RERUN."
)

DASO-Net — FINALIZE SEED-3 QC
PASS  Seed-3 terminal checkpoint
PASS  Seed-3 training manifest
PASS  Seed-3 exact-update manifest
PASS  Seed-3 QC settings
PASS  Seed-3 QC stage manifest
PASS  Metric summary JSON
PASS  Per-scene CSV
PASS  Oracle JSON
PASS  Oracle CSV
PASS  Counterfactual JSON
PASS  Seed-1 scene reference

Seed-3 terminal SHA: PASS
Seed-3 RNG identity: PASS
Seed-3 exact 2750 updates: PASS
Same ordered 1000-scene set as Seed 1: PASS

Headline values verified against canonical JSONs: PASS
All Seed-3 QC sanity flags: PASS

FINAL SEED 3 QC FINALIZATION COMPLETE
AP50:                    0.2591
AP75:                    0.0547
AP50:95:                 0.0966
BestF1:                  0.3953
Best threshold:          0.35

Oracle GT score:         0.2663
Oracle PeakRecall@8:     0.7751
Oracle BoxIoU:           0.5793
Oracle GT-box MaskIoU:   0.4712

CF exact lift:          +0.2552
CF local lift:          +0.3294
CF comp > clean exact:   0.9744
CF effect max:           0.4248
CF out

In [6]:
# ============================================================
# DASO-Net — AFTER-SEED3 / THREE-SEED EOD HANDOFF
#
# Date: 2026-09-18
#
# Contains:
#   - validated code
#   - frozen training recipe
#   - frozen multi-seed plan
#   - Seed 1 terminal + metadata + QC
#   - Seed 2 terminal + metadata + QC
#   - Seed 3 terminal + metadata + QC
#   - duration-selection summary
#   - learning-curve summary
#   - restore script
#   - file-by-file SHA manifest
#
# Does NOT contain:
#   - the large frozen dasonet-project dataset
#   - nested ZIP archives
#
# Tomorrow:
#   attach this handoff + dasonet-project,
#   restore,
#   then begin three-seed aggregate/formal evaluation.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
import zipfile
from pathlib import Path


# ============================================================
# 1. IDENTITY
# ============================================================

DATE_TAG = "20260918"

PACKAGE_NAME = (
    f"DASO-Net-AFTER-SEED3-EOD-{DATE_TAG}"
)

SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796c9cc301bd6fcc7c596fc290ad3a7e81d"
)

SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe5738d9c7cd5d8f818c62ad1e289df1ad2"
)

SEED3_SHA = (
    "be7b7941f2f62179e15074aa973bf6da0eb9018ec9133486512bd1f87153f447"
)

DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1"
)

SEED1 = 1782656963
SEED2 = 901395089
SEED3 = 252916394


# ============================================================
# 2. ROOTS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

STAGE_ROOT = (
    WORKING
    / PACKAGE_NAME
)

STAGE_WORKSPACE = (
    STAGE_ROOT
    / "workspace"
)

ZIP_PATH = (
    WORKING
    / f"{PACKAGE_NAME}.zip"
)

SHA_PATH = (
    WORKING
    / f"{PACKAGE_NAME}.sha256.txt"
)

STATE_PATH = (
    WORKING
    / f"{PACKAGE_NAME}-STATE.json"
)

INNER_MANIFEST = (
    STAGE_ROOT
    / "HANDOFF_MANIFEST.json"
)

INNER_STATE = (
    STAGE_ROOT
    / "AFTER_SEED3_STATE.json"
)

RESTORE_SCRIPT = (
    STAGE_ROOT
    / "RESTORE_AFTER_SEED3.py"
)

README = (
    STAGE_ROOT
    / "README_HANDOFF.txt"
)


# ============================================================
# 3. SOURCE PATHS
# ============================================================

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

MULTISEED_ROOT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

FREEZE_ROOT = (
    WORKING
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

BASE_CONFIG = (
    WORKING
    / "phase5c_baseline_curve1000updates_v1.json"
)

AFTER_SEED2_RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED2-RESTORE-AUDIT-v1.json"
)


# ------------------------------------------------------------
# Seed 1
# ------------------------------------------------------------

SEED1_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

SEED1_TERMINAL = (
    SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED1_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)


# ------------------------------------------------------------
# Seed 2
# ------------------------------------------------------------

SEED2_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)

SEED2_TERMINAL = (
    SEED2_RUN
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED2_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
)

SEED2_QC_SUMMARY = (
    SEED2_QC
    / "FINAL_SEED02_QC_SUMMARY.json"
)


# ------------------------------------------------------------
# Seed 3
# ------------------------------------------------------------

SEED3_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
)

SEED3_TERMINAL = (
    SEED3_RUN
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED3_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed03_s252916394_val1000_v1_fixedop005"
)

SEED3_QC_SUMMARY = (
    SEED3_QC
    / "FINAL_SEED03_QC_SUMMARY.json"
)


# ------------------------------------------------------------
# Development provenance
# ------------------------------------------------------------

DURATION_SUMMARY = (
    WORKING
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
    / "duration_selection_val1000_summary.json"
)

LEARNING_CURVE_SUMMARY = (
    WORKING
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

MULTISEED_PLAN = (
    MULTISEED_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)


# ============================================================
# 4. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def copy_file(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        dst,
    )


def copy_tree(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        src,
        dst,
        ignore=shutil.ignore_patterns(
            "__pycache__",
            "*.pyc",
            "*.pyo",
        ),
    )


def copy_run_compact(
    src_run: Path,
    dst_run: Path,
    terminal: Path,
) -> None:
    """
    Preserve:
      terminal checkpoint
      all small top-level run metadata

    Do not preserve:
      historical snapshot chain
      alternate .pt/.pth/.ckpt files
    """

    dst_run.mkdir(
        parents=True,
        exist_ok=True,
    )

    dst_snapshots = (
        dst_run
        / "snapshots"
    )

    dst_snapshots.mkdir(
        parents=True,
        exist_ok=True,
    )


    copy_file(
        terminal,
        dst_snapshots
        / terminal.name,
    )


    for path in sorted(
        src_run.iterdir()
    ):

        if not path.is_file():
            continue

        if path.suffix.lower() in {
            ".pt",
            ".pth",
            ".ckpt",
        }:
            continue

        copy_file(
            path,
            dst_run
            / path.name,
        )


def tree_manifest(
    root: Path,
):
    records = []

    for path in sorted(
        p
        for p in root.rglob("*")
        if p.is_file()
    ):

        if path == INNER_MANIFEST:
            continue

        records.append(
            {
                "relative_path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )

    return records


# ============================================================
# 5. SOURCE AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — AFTER-SEED3 / THREE-SEED EOD HANDOFF"
)

print(
    "============================================================"
)


required = [
    (
        CODE_ROOT,
        "Validated code tree",
    ),
    (
        MULTISEED_ROOT,
        "Frozen multi-seed plan directory",
    ),
    (
        FREEZE_ROOT,
        "Frozen training recipe",
    ),
    (
        BASE_CONFIG,
        "Source training config",
    ),
    (
        AFTER_SEED2_RESTORE_AUDIT,
        "After-Seed2 restore audit",
    ),

    (
        SEED1_TERMINAL,
        "Seed-1 terminal",
    ),
    (
        SEED1_QC,
        "Seed-1 QC",
    ),

    (
        SEED2_TERMINAL,
        "Seed-2 terminal",
    ),
    (
        SEED2_QC,
        "Seed-2 QC",
    ),
    (
        SEED2_QC_SUMMARY,
        "Seed-2 final QC summary",
    ),

    (
        SEED3_TERMINAL,
        "Seed-3 terminal",
    ),
    (
        SEED3_QC,
        "Seed-3 QC",
    ),
    (
        SEED3_QC_SUMMARY,
        "Seed-3 final QC summary",
    ),

    (
        DURATION_SUMMARY,
        "Duration-selection summary",
    ),
    (
        LEARNING_CURVE_SUMMARY,
        "Learning-curve summary",
    ),

    (
        DECODER,
        "Validated decoder",
    ),
    (
        MULTISEED_PLAN,
        "Frozen multi-seed plan JSON",
    ),
]


for path, label in required:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 6. CRITICAL HASH AUDIT
# ============================================================

if sha256_file(
    SEED1_TERMINAL
) != SEED1_SHA:

    raise RuntimeError(
        "Seed-1 terminal SHA mismatch."
    )


if sha256_file(
    SEED2_TERMINAL
) != SEED2_SHA:

    raise RuntimeError(
        "Seed-2 terminal SHA mismatch."
    )


if sha256_file(
    SEED3_TERMINAL
) != SEED3_SHA:

    raise RuntimeError(
        "Seed-3 terminal SHA mismatch."
    )


if sha256_file(
    DECODER
) != DECODER_SHA:

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


seed2_qc = load_json(
    SEED2_QC_SUMMARY
)

seed3_qc = load_json(
    SEED3_QC_SUMMARY
)


if seed2_qc.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Seed-2 QC is not PASS."
    )


if seed3_qc.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Seed-3 QC is not PASS."
    )


if not all(
    seed2_qc[
        "sanity_flags"
    ].values()
):

    raise RuntimeError(
        "Seed-2 QC sanity failure."
    )


if not all(
    seed3_qc[
        "sanity_flags"
    ].values()
):

    raise RuntimeError(
        "Seed-3 QC sanity failure."
    )


print()
print(
    "Seed-1 terminal SHA: PASS"
)

print(
    "Seed-2 terminal SHA: PASS"
)

print(
    "Seed-3 terminal SHA: PASS"
)

print(
    "Decoder SHA: PASS"
)

print(
    "Seed-2 final QC: PASS"
)

print(
    "Seed-3 final QC: PASS"
)

print(
    "THREE FINAL TRAINING SEEDS COMPLETE: PASS"
)


# ============================================================
# 7. CLEAN STAGING AREA
# ============================================================

if STAGE_ROOT.exists():

    shutil.rmtree(
        STAGE_ROOT
    )


for path in [
    ZIP_PATH,
    SHA_PATH,
    STATE_PATH,
]:

    if path.exists():
        path.unlink()


STAGE_WORKSPACE.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 8. COPY VALIDATED CORE
# ============================================================

copy_tree(
    CODE_ROOT,
    STAGE_WORKSPACE
    / CODE_ROOT.name,
)

copy_tree(
    MULTISEED_ROOT,
    STAGE_WORKSPACE
    / MULTISEED_ROOT.name,
)

copy_tree(
    FREEZE_ROOT,
    STAGE_WORKSPACE
    / FREEZE_ROOT.name,
)

copy_file(
    BASE_CONFIG,
    STAGE_WORKSPACE
    / BASE_CONFIG.name,
)

copy_file(
    AFTER_SEED2_RESTORE_AUDIT,
    STAGE_WORKSPACE
    / AFTER_SEED2_RESTORE_AUDIT.name,
)


# ============================================================
# 9. COPY SEED 1
# ============================================================

copy_run_compact(
    SEED1_RUN,
    STAGE_WORKSPACE
    / "dasonet_phase5b"
    / SEED1_RUN.name,
    SEED1_TERMINAL,
)

copy_tree(
    SEED1_QC,
    STAGE_WORKSPACE
    / "dasonet_final_qc"
    / SEED1_QC.name,
)


# ============================================================
# 10. COPY SEED 2
# ============================================================

copy_run_compact(
    SEED2_RUN,
    STAGE_WORKSPACE
    / "dasonet_phase5b"
    / SEED2_RUN.name,
    SEED2_TERMINAL,
)

copy_tree(
    SEED2_QC,
    STAGE_WORKSPACE
    / "dasonet_final_qc"
    / SEED2_QC.name,
)


# ============================================================
# 11. COPY SEED 3
# ============================================================

copy_run_compact(
    SEED3_RUN,
    STAGE_WORKSPACE
    / "dasonet_phase5b"
    / SEED3_RUN.name,
    SEED3_TERMINAL,
)

copy_tree(
    SEED3_QC,
    STAGE_WORKSPACE
    / "dasonet_final_qc"
    / SEED3_QC.name,
)


# ============================================================
# 12. COPY DEVELOPMENT PROVENANCE
# ============================================================

copy_file(
    DURATION_SUMMARY,
    STAGE_WORKSPACE
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
    / DURATION_SUMMARY.name,
)

copy_file(
    LEARNING_CURVE_SUMMARY,
    STAGE_WORKSPACE
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / LEARNING_CURVE_SUMMARY.name,
)


# ============================================================
# 13. INNER STATE
# ============================================================

state = {
    "handoff":
        "DASO-Net AFTER-SEED3 / THREE-SEED EOD",

    "date_tag":
        DATE_TAG,

    "final_multiseed_training":
        "COMPLETE",

    "final_seed_count":
        3,

    "seeds": [
        {
            "seed_index":
                1,

            "rng_seed":
                SEED1,

            "status":
                "COMPLETE",

            "successful_optimizer_updates":
                2750,

            "terminal_sha256":
                SEED1_SHA,

            "qc":
                "PASS",
        },

        {
            "seed_index":
                2,

            "rng_seed":
                SEED2,

            "status":
                "COMPLETE",

            "successful_optimizer_updates":
                2750,

            "terminal_sha256":
                SEED2_SHA,

            "qc":
                "PASS",

            "qc_summary_sha256":
                sha256_file(
                    SEED2_QC_SUMMARY
                ),
        },

        {
            "seed_index":
                3,

            "rng_seed":
                SEED3,

            "status":
                "COMPLETE",

            "successful_optimizer_updates":
                2750,

            "terminal_sha256":
                SEED3_SHA,

            "qc":
                "PASS",

            "qc_summary_sha256":
                sha256_file(
                    SEED3_QC_SUMMARY
                ),
        },
    ],

    "decoder_sha256":
        DECODER_SHA,

    "next_scientific_stage": [
        "Three-seed aggregate mean and SD",
        "Untouched FruitMP-Syn test evaluation",
        "Held-out clean FruitBG false-positive evaluation",
        "External RSC evaluation",
        "Tioman limited FTIR-linked evidence",
        "Talanta qualitative food/smartphone transfer",
        "Formal ablations",
        "Calibration/selective screening",
    ],

    "external_data_used_for_final_seed_selection":
        False,

    "best_seed_selection":
        False,

    "note":
        (
            "All three precommitted final seeds are preserved. "
            "Do not select or discard a seed based on QC results."
        ),
}


save_json(
    INNER_STATE,
    state,
)


# ============================================================
# 14. README
# ============================================================

README.write_text(
    """
DASO-Net AFTER-SEED3 / THREE-SEED EOD HANDOFF
==============================================

FINAL TRAINING STATUS
---------------------
Seed 1:
  RNG 1782656963
  2750 successful optimizer updates
  QC complete

Seed 2:
  RNG 901395089
  2750 successful optimizer updates
  QC PASS

Seed 3:
  RNG 252916394
  2750 successful optimizer updates
  QC PASS

POLICY
------
- All three seeds are retained.
- No best-seed selection.
- No final seed is discarded.
- Report individual seeds and mean +/- SD.
- External test sets were not used to select final seeds.

TOMORROW
--------
Attach:
  1. abinavprabhuv/dasonet-project
  2. the Kaggle dataset created from this handoff

Then:
  restore -> audit -> three-seed aggregate -> formal evaluations.

The large frozen project dataset is intentionally excluded from
this handoff.
""".strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 15. RESTORE SCRIPT FOR TOMORROW
# ============================================================

restore_source = r'''
from __future__ import annotations

import hashlib
import shutil
from pathlib import Path


PACKAGE_ROOT = Path(__file__).resolve().parent

SOURCE = (
    PACKAGE_ROOT
    / "workspace"
)

DEST = Path(
    "/kaggle/working"
)

EXPECTED = {
    "seed1":
        "e4c4b79ad4b8c6e18ba2b1e3d6cbe796c9cc301bd6fcc7c596fc290ad3a7e81d",

    "seed2":
        "19cdb834a46240916b2c8023e8c3afe5738d9c7cd5d8f818c62ad1e289df1ad2",

    "seed3":
        "be7b7941f2f62179e15074aa973bf6da0eb9018ec9133486512bd1f87153f447",
}


def sha256_file(path: Path) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def merge_tree(
    src: Path,
    dst: Path,
) -> None:

    dst.mkdir(
        parents=True,
        exist_ok=True,
    )

    for item in src.iterdir():

        target = (
            dst
            / item.name
        )

        if item.is_dir():

            merge_tree(
                item,
                target,
            )

        else:

            target.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(
                item,
                target,
            )


if not SOURCE.exists():

    raise RuntimeError(
        f"Workspace source missing: {SOURCE}"
    )


merge_tree(
    SOURCE,
    DEST,
)


terminals = {
    "seed1":
        DEST
        / "dasonet_phase5b"
        / "dasonet_final_seed01_s1782656963"
        / "snapshots"
        / "epoch_011_opt_002750.pt",

    "seed2":
        DEST
        / "dasonet_phase5b"
        / "dasonet_final_seed02_s901395089"
        / "snapshots"
        / "epoch_010_opt_002750.pt",

    "seed3":
        DEST
        / "dasonet_phase5b"
        / "dasonet_final_seed03_s252916394"
        / "snapshots"
        / "epoch_010_opt_002750.pt",
}


for name, path in terminals.items():

    if not path.exists():

        raise RuntimeError(
            f"{name} terminal missing after restore."
        )

    actual = sha256_file(
        path
    )

    if actual != EXPECTED[name]:

        raise RuntimeError(
            f"{name} terminal SHA mismatch."
        )


seed2_qc = (
    DEST
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
    / "FINAL_SEED02_QC_SUMMARY.json"
)

seed3_qc = (
    DEST
    / "dasonet_final_qc"
    / "seed03_s252916394_val1000_v1_fixedop005"
    / "FINAL_SEED03_QC_SUMMARY.json"
)


if not seed2_qc.exists():

    raise RuntimeError(
        "Seed-2 QC summary missing."
    )


if not seed3_qc.exists():

    raise RuntimeError(
        "Seed-3 QC summary missing."
    )


print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED3 RESTORE: PASS"
)

print(
    "============================================================"
)

print(
    "Seed 1 terminal: PASS"
)

print(
    "Seed 2 terminal: PASS"
)

print(
    "Seed 3 terminal: PASS"
)

print(
    "Seed 2 QC: PASS"
)

print(
    "Seed 3 QC: PASS"
)

print(
    "All three final seeds restored."
)

print(
    "Next: three-seed aggregate and formal evaluation."
)
'''


RESTORE_SCRIPT.write_text(
    restore_source.strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 16. BUILD FILE MANIFEST
# ============================================================

records = tree_manifest(
    STAGE_ROOT
)


manifest = {
    "package_name":
        PACKAGE_NAME,

    "file_count":
        len(
            records
        ),

    "files":
        records,
}


save_json(
    INNER_MANIFEST,
    manifest,
)


all_stage_files = [
    path
    for path in STAGE_ROOT.rglob("*")
    if path.is_file()
]


print()
print(
    "Staged files:",
    len(
        all_stage_files
    ),
)


# ============================================================
# 17. CREATE ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for path in sorted(
        all_stage_files
    ):

        relative = (
            Path(
                PACKAGE_NAME
            )
            / path.relative_to(
                STAGE_ROOT
            )
        )


        compression = (
            zipfile.ZIP_STORED
            if path.suffix.lower()
            in {
                ".pt",
                ".pth",
                ".ckpt",
            }
            else zipfile.ZIP_DEFLATED
        )


        zf.write(
            path,
            arcname=str(
                relative
            ),
            compress_type=compression,
        )


# ============================================================
# 18. CRC CHECK
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as zf:

    bad = zf.testzip()


    if bad is not None:

        raise RuntimeError(
            f"ZIP CRC failure: {bad}"
        )


# ============================================================
# 19. FULL EXTRACT / FILE HASH AUDIT
# ============================================================

with tempfile.TemporaryDirectory() as tmp:

    tmp_root = Path(
        tmp
    )


    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as zf:

        zf.extractall(
            tmp_root
        )


    extracted_root = (
        tmp_root
        / PACKAGE_NAME
    )


    extracted_manifest = load_json(
        extracted_root
        / "HANDOFF_MANIFEST.json"
    )


    for record in extracted_manifest[
        "files"
    ]:

        path = (
            extracted_root
            / record[
                "relative_path"
            ]
        )


        if not path.exists():

            raise RuntimeError(
                "Extracted file missing:\n"
                f"{path}"
            )


        if (
            path.stat().st_size
            != int(
                record[
                    "size_bytes"
                ]
            )
        ):

            raise RuntimeError(
                "Extracted size mismatch:\n"
                f"{path}"
            )


        if (
            sha256_file(
                path
            )
            != record[
                "sha256"
            ]
        ):

            raise RuntimeError(
                "Extracted SHA mismatch:\n"
                f"{path}"
            )


# ============================================================
# 20. OUTER STATE + SHA
# ============================================================

zip_sha = sha256_file(
    ZIP_PATH
)


outer_state = {
    "package":
        ZIP_PATH.name,

    "package_sha256":
        zip_sha,

    "package_size_bytes":
        ZIP_PATH.stat().st_size,

    "crc":
        "PASS",

    "staged_file_sha_audit":
        "PASS",

    "three_final_seeds_complete":
        True,

    "seed1_terminal_sha256":
        SEED1_SHA,

    "seed2_terminal_sha256":
        SEED2_SHA,

    "seed3_terminal_sha256":
        SEED3_SHA,

    "seed2_qc_summary_sha256":
        sha256_file(
            SEED2_QC_SUMMARY
        ),

    "seed3_qc_summary_sha256":
        sha256_file(
            SEED3_QC_SUMMARY
        ),

    "multiseed_plan_sha256":
        sha256_file(
            MULTISEED_PLAN
        ),

    "decoder_sha256":
        DECODER_SHA,

    "next_stage":
        (
            "three-seed aggregate and formal final evaluation"
        ),

    "training_remaining":
        False,

    "best_seed_selection":
        False,

    "required_new_notebook_inputs":
        [
            "abinavprabhuv/dasonet-project",
            PACKAGE_NAME,
        ],
}


save_json(
    STATE_PATH,
    outer_state,
)


SHA_PATH.write_text(
    f"{zip_sha}  {ZIP_PATH.name}\n",
    encoding="utf-8",
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED3 / THREE-SEED EOD HANDOFF: PASS"
)

print(
    "============================================================"
)

print()

print(
    "ZIP:"
)

print(
    ZIP_PATH
)

print()

print(
    "Size:"
)

print(
    f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MiB"
)

print()

print(
    "SHA256:"
)

print(
    zip_sha
)

print()

print(
    "SHA file:"
)

print(
    SHA_PATH
)

print()

print(
    "State file:"
)

print(
    STATE_PATH
)

print()

print(
    "CRC: PASS"
)

print(
    "Extracted file-by-file SHA audit: PASS"
)

print()

print(
    "Seed 1: COMPLETE + PRESERVED"
)

print(
    "Seed 2: COMPLETE + QC PASS + PRESERVED"
)

print(
    "Seed 3: COMPLETE + QC PASS + PRESERVED"
)

print()

print(
    "ALL THREE FINAL TRAINING SEEDS: COMPLETE"
)

print(
    "NO FURTHER FINAL-SEED TRAINING REMAINS."
)

print()

print(
    "DOWNLOAD THESE THREE FILES:"
)

print(
    f"1. {ZIP_PATH.name}"
)

print(
    f"2. {SHA_PATH.name}"
)

print(
    f"3. {STATE_PATH.name}"
)

print()

print(
    "AFTER DOWNLOADING: STOP FOR TODAY."
)

print(
    "Tomorrow: restore -> three-seed aggregate -> formal evaluation."
)

DASO-Net — AFTER-SEED3 / THREE-SEED EOD HANDOFF
PASS  Validated code tree
PASS  Frozen multi-seed plan directory
PASS  Frozen training recipe
PASS  Source training config
PASS  After-Seed2 restore audit
PASS  Seed-1 terminal
PASS  Seed-1 QC
PASS  Seed-2 terminal
PASS  Seed-2 QC
PASS  Seed-2 final QC summary
PASS  Seed-3 terminal
PASS  Seed-3 QC
PASS  Seed-3 final QC summary
PASS  Duration-selection summary
PASS  Learning-curve summary
PASS  Validated decoder
PASS  Frozen multi-seed plan JSON

Seed-1 terminal SHA: PASS
Seed-2 terminal SHA: PASS
Seed-3 terminal SHA: PASS
Decoder SHA: PASS
Seed-2 final QC: PASS
Seed-3 final QC: PASS
THREE FINAL TRAINING SEEDS COMPLETE: PASS

Staged files: 103

DASO-NET AFTER-SEED3 / THREE-SEED EOD HANDOFF: PASS

ZIP:
/kaggle/working/DASO-Net-AFTER-SEED3-EOD-20260918.zip

Size:
55.29 MiB

SHA256:
21aca1cc30f70c3229f8f9c357e4e8ebf7e711804ffefacae325542b269c610c

SHA file:
/kaggle/working/DASO-Net-AFTER-SEED3-EOD-20260918.sha256.txt

State file:
/kaggle/work